# Pairs Trading with a Regime-Aware Adaptive Z-Score (KF–GARCH–HMM)

Companion notebook to the Master's thesis *High-Dimensional Statistical Arbitrage: An Integrated KF-HMM-GARCH Framework for Dynamic Multi-Asset Trading* (University of Bologna, 2025/2026). It implements the bivariate branch of the empirical part (Part IV, Section 12).

The pipeline discovers cointegrated pairs on rolling windows, models each spread dynamically and compares three nested specifications of the trading signal:

| Spec | Hedge ratio | z-score denominator | Regime gate | Label in plots |
|---|---|---|---|---|
| `ols`  | static OLS, estimated on formation | rolling sample std | none | OLS |
| `kf`   | Kalman Filter, time-varying | $\sqrt{f_t + \hat\sigma^2_{S,t}}$ (KF innovation variance + GARCH) | none | KF+GARCH |
| `full` | Kalman Filter, time-varying | $\sqrt{f_t + \hat\sigma^2_{S,t}}$ | HMM, trade only if $\phi_t > \phi^*$ | KF+GARCH+HMM |

`ols → kf` isolates the dynamic hedge ratio plus the GARCH denominator; `kf → full` isolates the HMM regime filter.

**Pipeline stages** (each stage caches its output in the HDF5 store and is skipped on re-runs unless forced):

$$\text{T1 screening} \to \text{T2 KF + GARCH} \to \text{T3 HMM regimes} \to \text{T4 signals} \to \text{T5 thresholds (CPCV, DSR, PBO)} \to \text{quality ranking} \to \text{backtest}$$

**Out-of-sample discipline.** Every model is estimated on the formation window only and applied causally to the trading window. The thresholds $(z^*, z^{\text{exit}})$ are optimized once on the development sample and then frozen for the holdout. A second, independent experiment repeats the whole protocol with a development sample ending in June 2007 and a holdout restricted to the 2008–2009 crisis.

### Data
The notebook reads a single HDF5 file (`Config.hdf_file`, not distributed with this repository) with the following keys:

| Key | Content |
|---|---|
| `/stocks/prices/adjusted` | daily adjusted prices, MultiIndex `(date, ticker)`, columns `close`, `volume` |
| `/etfs/prices/adjusted` | same layout for ETFs |
| `/reference/sp500_sectors` | column `sector` (GICS), indexed by ticker |
| `/etfs/classification` | column `asset_class`, indexed by ticker (optional) |

All outputs are written back to the same file under separate keys (`selection/…`, `spreads/…`, `regimes/…`, `signals/…`, `params/…`), with a namespace prefix for the holdout runs (`holdout/…`, `crisis2008_dev/…`, `crisis2008_holdout/…`).

> **Survivorship bias.** The equity universe used in the thesis is built from current constituents filled backwards in time (see the audit in Section 11.5). All results are therefore an upper bound on achievable real-time performance.

### How to run
Run the cells top to bottom. Sections 1–10 only define functions; Section 12 executes the experiments. The basket notebook (`Mstatarb_sandbox.ipynb`) reuses the thresholds written by this notebook, so run this one first on the same HDF5 file.

## 0 · Setup
Mount Google Drive (where the HDF5 file lives) and install `arch`. Outside Colab, skip the first two lines and install the dependencies from `requirements.txt`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!pip install arch -q

In [ ]:
from __future__ import annotations
import os, warnings, itertools; warnings.filterwarnings('ignore')
from dataclasses import dataclass
from itertools import combinations
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss, coint

ANN = np.sqrt(252)   # annualization factor for daily Sharpe ratios

### 0b · Plot style
Serif fonts, `deep` palette, 300 dpi export. Colour convention used in every figure: OLS = near-black, KF+GARCH = blue, KF+GARCH+HMM = red.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(
    style='whitegrid', context='paper', font='serif',
    rc={
        'axes.spines.top':   False,
        'axes.spines.right': False,
        'grid.linestyle':    '--',
        'grid.alpha':        0.4,
        'axes.titlesize':    11,
        'axes.labelsize':    10,
        'xtick.labelsize':   8.5,
        'ytick.labelsize':   8.5,
        'legend.fontsize':   8.5,
        'figure.dpi':        150,
        'savefig.dpi':       300,
        'savefig.bbox':      'tight',
    }
)
plt.rcParams['axes.titleweight'] = 'bold'

PALETTE = sns.color_palette('deep', 6)
# Colour map; keys are kept for backward compatibility with the plotting code.
C = {
    'bull':   PALETTE[0],            # blue   -> KF
    'gfc':    PALETTE[3],            # red    -> full
    'normal': (0.15, 0.15, 0.15),    # near-black -> OLS
}
OUTPUT_DIR = './'   # where figures and CSV tables are saved

## 1 · Configuration
Every tunable parameter lives in `Config`. The comments group them by pipeline stage (T1–T5). The defaults are the values used in the thesis.

In [ ]:
@dataclass
class Config:
    """All pipeline parameters. Defaults reproduce the thesis setup."""
    # IO
    hdf_file: str = '/content/drive/MyDrive/data_ip_2026_v2.h5'
    work_local: bool = False

    # universe
    use_etfs: bool = True
    use_sp500: bool = True

    # liquidity / data-quality filter applied before screening
    min_price: float = 5.0
    min_dollar_vol: float = 1e6
    max_missing_frac: float = 0.02
    min_obs_frac: float = 0.95

    # walk-forward windows
    formation_years: float = 3.0
    trading_years: float = 1.0
    embargo_days: int = 10
    dev_end: str = '2019-12-31'
    holdout_start: str = '2020-01-01'

    # screening (T1)
    adf_alpha: float = 0.05
    kpss_alpha: float = 0.05
    fdr_q: float = 0.10
    corr_max: float = 0.90
    stability_alpha: float = 0.10
    half_life_min: float = 10.0
    half_life_max: float = 120.0
    half_life_target: float = 25.0
    hurst_max: float = 0.50
    vr_max: float = 1.00
    top_pairs_per_group: int = 10
    max_pairs_per_window: int = 25
    max_pairs_per_asset: int = 3

    # spreads (T2)
    kf_kappa: float = 1e6
    kf_burnin: int = 20
    kf_starts: int = 6
    snr_lo: float = 1e-1
    snr_hi: float = 1e4
    garch_scale: float = 1e4
    beta_dstd_max: float = 0.20
    garch_vols: tuple = ('GARCH', 'GJR', 'EGARCH')
    garch_dists: tuple = ('normal', 't', 'skewt')

    # regimes (T3)
    hmm_restarts: int = 6
    hmm_iters: int = 80
    rv_window: int = 21
    var_floor: float = 1e-4

    # signals (T4)
    std_win: int = 252
    std_minp: int = 60

    # thresholds / strategy (T5)
    z_stop: float = 4.0
    max_hold_mult: int = 5
    phi_star: float = 0.65
    grid_zstar: tuple = (1.5, 2.0, 2.5, 3.0, 3.5)
    grid_zexit: tuple = (0.25, 0.5, 0.75, 1.0)
    cost_bps: float = 5.0
    cscv_blocks: int = 8

    # backtest
    top_n: int = 5
    seed: int = 0
    holdout_end: str = None   # optional upper bound for a bounded holdout (e.g. the 2008-2009 crisis)

## 2 · Universe and I/O
ETFs are grouped into economic buckets (sector, country, commodity, bonds, …) and stocks by GICS sector; candidate pairs are only formed **within** a bucket. Broad-market beta ETFs are excluded from pairing (`NO_PAIR_BUCKETS`) because they are near-duplicates of each other.

`Store` wraps the HDF5 file. Price data are read-only; every stage writes its output to its own key, optionally under a namespace (`ns`) so that holdout runs never overwrite the development cache.

In [ ]:
# Economic buckets for ETFs. Tickers not listed fall back to the provider's asset class.
_ETF_TAXO = {
    'broad_beta': {'SPY','IVV','VOO','VTI','QQQ','IWM','IWB','IJH','IJR','MDY','ITOT',
                   'SCHB','VV','OEF','RSP','SCHX','SPLG','IWD','IWF','VTV','VUG'},
    'sector_us': {'XLK','XLF','XLV','XLE','XLI','XLY','XLP','XLU','XLRE','XLB','XLC',
                  'VGT','VFH','VHT','VDE','VIS','VCR','VDC','VPU','VNQ','VAW','VOX'},
    'country': {'EWJ','EWG','EWU','EWQ','EWI','EWP','EWD','EWN','EWL','EWA','EWC','EWZ',
                'EWW','EWY','EWT','EWS','EWH','EWM','MCHI','FXI','INDA','EPOL','ECH','EZA','FM'},
    'regional': {'EFA','VEA','IEFA','EEM','VWO','IEMG','VGK','IEV','EZU','EPP','VPL','AAXJ'},
    'commodity': {'GLD','SLV','IAU','SGOL','SIVR','PPLT','PALL','USO','BNO','UNG','DBC',
                  'GSG','PDBC','DBA','CORN','WEAT','CPER','DBO','DBB','UGA'},
    'currency': {'UUP','UDN','FXE','FXY','FXB','FXF','FXC','FXA'},
    'bond_tsy': {'SHY','IEI','IEF','TLT','GOVT','VGIT','VGLT','VGSH','BIL','SHV','SGOV'},
    'bond_credit': {'LQD','VCIT','IGIB','HYG','JNK','USIG'},
}
_T2B = {t: b for b, ts in _ETF_TAXO.items() for t in ts}
NO_PAIR_BUCKETS = {'etf::broad_beta'}
def etf_bucket(t, coarse):
    b = _T2B.get(t); return f"etf::{b}" if b else f"etf::{coarse}"


# ═════════════════════════════════════════════════════════════════════════════
# I/O: the price database is read-only; each stage writes to its own key
# ═════════════════════════════════════════════════════════════════════════════
class Store:
    """Thin wrapper around the HDF5 file (prices in, stage outputs out)."""
    def __init__(self, cfg: Config, ns=None):
        # ns: namespace for the outputs (e.g. 'holdout'). Price inputs are never
        # namespaced; only the stage outputs are isolated.
        self.cfg = cfg; self.path = cfg.hdf_file; self.ns = ns
        if cfg.work_local:
            import shutil; loc = '/content/_prices.h5'
            if not os.path.exists(loc):
                print(f'Copying DB -> {loc}'); shutil.copy(cfg.hdf_file, loc)
            self.path = loc

    def _k(self, key):
        return f"{self.ns}/{key.strip('/')}" if self.ns else key

    def keys(self):
        with pd.HDFStore(self.path, 'r') as s: return list(s.keys())

    def has(self, key):
        return ('/'+self._k(key).strip('/')) in self.keys()

    def read(self, key):
        with pd.HDFStore(self.path, 'r') as s: return s[self._k(key)]

    def write(self, key, df, data_columns=None):
        k = self._k(key)
        with pd.HDFStore(self.path, 'a') as s:
            if k in s: s.remove(k)
            s.put(k, df, format='table', data_columns=data_columns)

    def load_universe(self, tickers=None):
        """Return (close, groups, volume): wide price/volume panels and ticker -> bucket map."""
        parts, groups = [], {}
        with pd.HDFStore(self.path, 'r') as s:
            keys = set(s.keys())
            def cv(key):
                p = s[key]
                if tickers is not None:
                    p = p.loc[p.index.get_level_values('ticker').isin(set(tickers))]
                return (p['close'].unstack('ticker') if len(p) else None,
                        p['volume'].unstack('ticker') if (len(p) and 'volume' in p.columns) else None)
            vol_parts = []
            if self.cfg.use_etfs and '/etfs/prices/adjusted' in keys:
                cl, vo = cv('/etfs/prices/adjusted')
                if cl is not None:
                    parts.append(cl); vol_parts.append(vo)
                    ec = s['/etfs/classification']['asset_class'] if '/etfs/classification' in keys else pd.Series(dtype=str)
                    for t in cl.columns: groups[t] = etf_bucket(t, ec.get(t, 'us_listed'))
            if self.cfg.use_sp500 and '/stocks/prices/adjusted' in keys:
                cl, vo = cv('/stocks/prices/adjusted')
                if cl is not None:
                    sec = s['/reference/sp500_sectors']['sector'] if '/reference/sp500_sectors' in keys else pd.Series(dtype=str)
                    sp = [t for t in cl.columns if t in sec.index] or list(cl.columns)
                    cl = cl[sp]; vo = vo[sp] if vo is not None else None
                    parts.append(cl); vol_parts.append(vo)
                    for t in cl.columns: groups[t] = f"gics::{sec.get(t,'Unknown')}"
        close = pd.concat(parts, axis=1).sort_index()
        close = close.loc[:, ~close.columns.duplicated()]
        vol = None
        if vol_parts and all(v is not None for v in vol_parts):
            vol = pd.concat(vol_parts, axis=1).sort_index()
            vol = vol.loc[:, ~vol.columns.duplicated()]
        return close, groups, vol

## 3 · Walk-forward windows
Each window is `formation (3y) → embargo (10d) → trading (1y)`, rolled forward by one trading year. The window id used throughout the notebook is `"<formation start>_<trading end>"`.

A window belongs to the **development** sample if its trading period ends on or before `dev_end`, and to the **holdout** if its trading period ends on or after `holdout_start` (and before `holdout_end`, when set). Note that the classification uses the *end* of the trading year: the first holdout window of the main experiment (trading end 2020-01-16) therefore trades mostly during 2019.

In [ ]:
def walk_forward(index, cfg: Config, holdout=False):
    """Yield (formation_start, formation_end, embargo_end, trading_end) tuples.

    Development windows must end by cfg.dev_end; holdout windows must end on or
    after cfg.holdout_start (and by cfg.holdout_end, if set)."""
    end = pd.Timestamp('2100-01-01') if holdout else pd.Timestamp(cfg.dev_end)
    idx = index[index <= end]
    if len(idx) == 0: return
    fdays = int(cfg.formation_years*252); tdays = int(cfg.trading_years*252); emb = cfg.embargo_days
    holdout_end = pd.Timestamp(cfg.holdout_end) if (holdout and getattr(cfg, 'holdout_end', None)) else None
    start = 0
    while start + fdays + emb + tdays <= len(idx):
        f0, f1 = idx[start], idx[start+fdays-1]
        e1 = idx[start+fdays-1+emb]
        t1 = idx[min(start+fdays-1+emb+tdays, len(idx)-1)]
        in_holdout_range = (t1 >= pd.Timestamp(cfg.holdout_start)) and (holdout_end is None or t1 <= holdout_end)
        if (not holdout) or in_holdout_range:
            yield (f0, f1, e1, t1)
        start += tdays

## 4 · Stage T1 — Pair screening
For each window, on formation data only:

1. **Liquidity filter** — minimum price, median dollar volume, missing-data limits.
2. **Order of integration** — each log-price must be $I(1)$ (ADF fails to reject in levels, KPSS rejects in levels, ADF rejects in differences).
3. **Cointegration** — Engle–Granger test on every intra-bucket pair, with Benjamini–Hochberg FDR control at $q = 0.10$.
4. **Stability** — the relationship must also be cointegrated on both halves of the formation window.
5. **Tradability** — return correlation below 0.90, bias-corrected half-life in [10, 120] days, Hurst exponent below 0.5, variance ratio below 1.

Survivors are ranked by the screening score $|\ln(t_{1/2}/25)| + H + \mathrm{VR}$ (lower is better), capped at 10 pairs per bucket, 25 per window and 3 per asset.

In [ ]:
def quality_filter(close_w, vol_w, cfg):
    keep = []
    for t in close_w.columns:
        s = close_w[t]
        if s.notna().mean() < cfg.min_obs_frac: continue
        if s.isna().mean() > cfg.max_missing_frac: continue
        if (s.dropna() < cfg.min_price).mean() > 0.10: continue
        if vol_w is not None and t in vol_w.columns:
            dv = (s*vol_w[t]).median()
            if not np.isfinite(dv) or dv < cfg.min_dollar_vol: continue
        keep.append(t)
    return keep

def build_groups(close_w, tickers, groups0, cfg):
    groups = {}
    for t in (x for x in tickers if x in groups0):
        groups.setdefault(groups0[t], []).append(t)
    return {g: ts for g, ts in groups.items() if len(ts) >= 2}

def is_I1(logprice, cfg):
    x = np.asarray(logprice.dropna(), float)
    if len(x) < 100: return False
    try:
        adf_lvl = adfuller(x, autolag='AIC')[1]
        adf_dif = adfuller(np.diff(x), autolag='AIC')[1]
        kpss_lvl = kpss(x, regression='c', nlags='auto')[1]
    except Exception:
        return False
    return bool((adf_lvl > cfg.adf_alpha) and (kpss_lvl < cfg.kpss_alpha) and (adf_dif < cfg.adf_alpha))

def align_pair(log_w, a, b):
    df = pd.concat([log_w[a], log_w[b]], axis=1).dropna()
    return df.iloc[:, 0].values, df.iloc[:, 1].values

def eg_pvalue(y1, y2):
    if len(y1) < 60: return 1.0
    try: return coint(y1, y2)[1]
    except Exception: return 1.0

def benjamini_hochberg(pvals, q):
    p = np.asarray(pvals, float); m = len(p)
    if m == 0: return np.zeros(0, bool)
    order = np.argsort(p); passed = p[order] <= q*(np.arange(1, m+1)/m)
    mask = np.zeros(m, bool)
    if passed.any(): mask[order[:np.max(np.where(passed)[0])+1]] = True
    return mask

def is_stable(y1, y2, alpha):
    n = len(y1); h = n//2
    if h < 60: return True
    return (eg_pvalue(y1[:h], y2[:h]) < alpha) and (eg_pvalue(y1[h:], y2[h:]) < alpha)

def _fit_ar1(s):
    s = np.asarray(s, float); x, y = s[:-1], s[1:]
    X = np.column_stack([np.ones_like(x), x]); b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return b[1], b[0]

def half_life(spread, T, dt=1.0):
    """AR(1) half-life with Kendall's small-sample bias correction of phi."""
    phi, _ = _fit_ar1(spread); phi = phi + (1+3*phi)/T
    return np.inf if not (0 < phi < 1) else np.log(2)/(-np.log(phi)/dt)

def hurst(x, max_lag=60):
    x = np.asarray(x, float); lags = np.arange(8, max_lag); rs = []
    for n in lags:
        m = len(x)//n
        if m < 1: break
        blk = x[:m*n].reshape(m, n); ch = blk - blk.mean(1, keepdims=True)
        R = np.maximum.accumulate(ch, 1).max(1) - np.minimum.accumulate(ch, 1).min(1)
        S = blk.std(1); rs.append(np.mean(R[S > 0]/S[S > 0]))
    lags = lags[:len(rs)]
    return np.polyfit(np.log(lags), np.log(rs), 1)[0]

def variance_ratio(x, q=5):
    x = np.asarray(x, float)
    return (x[q:]-x[:-q]).var(ddof=1)/(q*np.diff(x).var(ddof=1))

def tradability(y1, y2, cfg):
    """Tradability filters on the OLS spread; returns the pair statistics or None."""
    y1, y2 = np.asarray(y1, float), np.asarray(y2, float)
    rho = np.corrcoef(np.diff(y1), np.diff(y2))[0, 1]
    if not np.isfinite(rho) or rho > cfg.corr_max: return None
    beta = np.polyfit(y2, y1, 1)[0]; spread = y1 - beta*y2
    hl = half_life(spread, len(spread))
    if not (cfg.half_life_min <= hl <= cfg.half_life_max): return None
    H = hurst(spread); VR = variance_ratio(spread)
    if H > cfg.hurst_max or VR > cfg.vr_max: return None
    score = abs(np.log(hl/cfg.half_life_target)) + H + VR
    return dict(beta=beta, rho=rho, half_life=hl, hurst=H, vr=VR, score=score)

def cap_window(rows, cfg):
    """Keep the best-scoring pairs subject to per-window and per-asset caps."""
    rows = sorted(rows, key=lambda r: r['score']); cnt, out = {}, []
    for r in rows:
        if len(out) >= cfg.max_pairs_per_window: break
        if cnt.get(r['asset1'], 0) >= cfg.max_pairs_per_asset: continue
        if cnt.get(r['asset2'], 0) >= cfg.max_pairs_per_asset: continue
        out.append(r); cnt[r['asset1']] = cnt.get(r['asset1'], 0)+1; cnt[r['asset2']] = cnt.get(r['asset2'], 0)+1
    return out


# ═════════════════════════════════════════════════════════════════════════════
# STAGE T1 — SCREENING (idempotent: reuses the cached result unless force=True)
# ═════════════════════════════════════════════════════════════════════════════
def screen(cfg: Config, store: Store, force=False, holdout=False, verbose=True):
    """T1: run the screening funnel on every walk-forward window -> selection/pairs."""
    if store.has('selection/pairs') and not force:
        pairs_df = store.read('selection/pairs')
        if verbose: print(f"[screen] from cache: pairs={len(pairs_df)} (force=False)")
        return pairs_df

    close, groups0, vol = store.load_universe()
    logclose = np.log(close)
    if verbose:
        print(f"[screen] universe: {close.shape[1]} tickers | "
              f"{close.index.min().date()}–{close.index.max().date()}"
              f"{' | HOLDOUT (trading end >= ' + cfg.holdout_start + ')' if holdout else ''}")
    pairs_out = []
    for (f0, f1, e1, t1) in walk_forward(close.index, cfg, holdout=holdout):
        win = f"{f0.date()}_{t1.date()}"
        close_w, log_w = close.loc[f0:f1], logclose.loc[f0:f1]
        vol_w = vol.loc[f0:f1] if vol is not None else None
        univ = quality_filter(close_w, vol_w, cfg)
        groups = build_groups(close_w, univ, groups0, cfg)
        i1 = {t: is_I1(log_w[t], cfg) for g in groups.values() for t in g}
        window_rows = []
        for g, ts in groups.items():
            if g in NO_PAIR_BUCKETS: continue
            ts1 = [t for t in ts if i1.get(t, False)]
            if len(ts1) < 2: continue
            cand = list(combinations(ts1, 2))
            aligned = {(a, b): align_pair(log_w, a, b) for a, b in cand}
            pvals = [eg_pvalue(*aligned[(a, b)]) for a, b in cand]
            mask = benjamini_hochberg(pvals, cfg.fdr_q)
            rows_g = []
            for (a, b), pv, ok in zip(cand, pvals, mask):
                if not ok: continue
                ya, yb = aligned[(a, b)]
                if not is_stable(ya, yb, cfg.stability_alpha): continue
                tr = tradability(ya, yb, cfg)
                if tr: rows_g.append(dict(window=win, group=g, asset1=a, asset2=b, pvalue=pv, **tr))
            rows_g = sorted(rows_g, key=lambda r: r['score'])[:cfg.top_pairs_per_group]
            window_rows.extend(rows_g)
        pairs_out.extend(cap_window(window_rows, cfg))
        if verbose:
            print(f"  [{win}] univ={len(univ)} groups={len(groups)} "
                  f"pairs={sum(p['window']==win for p in pairs_out)}")
    pairs_df = pd.DataFrame(pairs_out)
    store.write('selection/pairs', pairs_df, data_columns=['window', 'group', 'asset1', 'asset2'])
    if verbose: print(f"[screen] total pairs={len(pairs_df)}")
    return pairs_df

## 5 · Stage T2 — Spread modelling: Kalman Filter + GARCH
**Kalman Filter.** The hedge ratio and intercept follow random walks, $y_{1,t} = \alpha_t + \beta_t y_{2,t} + \varepsilon_t$. The three hyperparameters $(\sigma^2_\varepsilon, q_\alpha, q_\beta)$ are estimated by maximum likelihood (prediction-error decomposition, L-BFGS-B with several starting points) **on the formation window only**, reparametrized through the signal-to-noise ratio. The filter is then run causally over the whole window; the spread is the one-step-ahead innovation $v_t$ with variance $f_t$.

**GARCH.** A conditional-variance model is selected by BIC on the formation innovations over the grid {GARCH, GJR, EGARCH} × {normal, t, skew-t}. Its parameters are frozen (`arch_model.fix`) and applied forward, so the variance $\hat\sigma^2_{S,t}$ is causal. Ljung–Box and Engle–Ng sign-bias p-values are stored as diagnostics. If `arch` is not installed, a Gaussian GARCH(1,1) fitted by Nelder–Mead is used instead.

**Executability filter.** A pair is kept for trading only if the daily hedge-ratio changes are small enough to rebalance: $\mathrm{std}(\Delta\beta_t) \le 0.20$ (`tradable_hedge`).

In [ ]:
from scipy.optimize import minimize as _minimize
from scipy.stats import f as _f_dist
try:
    from arch import arch_model as _arch_model
    _HAS_ARCH = True
    try:
        from arch.utility.exceptions import ConvergenceWarning as _ArchConv
        warnings.simplefilter('ignore', _ArchConv)
    except Exception:
        pass
except Exception:
    _HAS_ARCH = False

_VOL_KW = {'GARCH': dict(vol='Garch', p=1, o=0, q=1),
           'GJR':   dict(vol='Garch', p=1, o=1, q=1),
           'EGARCH':dict(vol='EGARCH',p=1, o=1, q=1)}

# ---- Kalman Filter (random-walk intercept and hedge ratio) ----
def kf_filter(y1, y2, sig_eps2, q_alpha, q_beta, kappa=1e6):
    """Filtered intercept, hedge ratio, innovations v_t and innovation variances f_t."""
    y1, y2 = np.asarray(y1, float), np.asarray(y2, float); T = len(y1)
    a = np.zeros(2); P = kappa*np.eye(2); Q = np.diag([q_alpha, q_beta])
    al = np.empty(T); be = np.empty(T); v = np.empty(T); f = np.empty(T)
    for t in range(T):
        z = np.array([1.0, y2[t]]); vt = y1[t]-z@a; ft = z@P@z+sig_eps2
        K = (P@z)/ft; a = a+K*vt; P = (np.eye(2)-np.outer(K, z))@P
        al[t], be[t], v[t], f[t] = a[0], a[1], vt, ft; P = P+Q
    return al, be, v, f

def _kf_nll(theta, y1, y2, burn):
    sig = np.exp(theta[0]); qa = sig*np.exp(theta[1]); qb = sig*np.exp(theta[2])
    _, _, v, f = kf_filter(y1, y2, sig, qa, qb); f, v = f[burn:], v[burn:]
    if np.any(f <= 0) or not np.all(np.isfinite(f)): return 1e12
    return 0.5*np.sum(np.log(2*np.pi*f)+v**2/f)

def fit_kf(y1, y2, cfg):
    """MLE of (sig_eps2, q_alpha, q_beta) with multi-start L-BFGS-B (formation data only)."""
    y1, y2 = np.asarray(y1, float), np.asarray(y2, float)
    b1, b0 = np.polyfit(y2, y1, 1); s0 = max(np.var(y1-(b1*y2+b0)), 1e-10); L = np.log
    bounds = [(L(1e-12), L(1e-1)), (L(1e-3), L(1e4)), (L(cfg.snr_lo), L(cfg.snr_hi))]
    starts = [[L(s0),L(1.0),L(10.0)], [L(1e-4),L(1.0),L(100.0)], [L(1e-6),L(1.0),L(700.0)],
              [L(1e-8),L(1.0),L(700.0)], [L(s0*0.1),L(10.0),L(50.0)], [L(1e-6),L(0.5),L(5.0)]][:cfg.kf_starts]
    best = None
    for x0 in starts:
        x0 = [min(max(xi, lo), hi) for xi, (lo, hi) in zip(x0, bounds)]
        try:
            r = _minimize(_kf_nll, x0, args=(y1, y2, cfg.kf_burnin), method='L-BFGS-B', bounds=bounds)
            if best is None or r.fun < best.fun: best = r
        except Exception: continue
    if best is None: return 1e-6, 1e-6, 1e-6
    sig = float(np.exp(best.x[0]))
    return sig, sig*float(np.exp(best.x[1])), sig*float(np.exp(best.x[2]))

# ---- GARCH selection + diagnostics ----
def select_garch(vs_form, cfg):
    """BIC selection over volatility models x innovation distributions."""
    if not _HAS_ARCH: return None
    best = None
    for vol in cfg.garch_vols:
        for dist in cfg.garch_dists:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    res = _arch_model(vs_form, mean='Zero', dist=dist, **_VOL_KW[vol]).fit(disp='off')
                if best is None or res.bic < best['bic']:
                    best = dict(vol=vol, dist=dist, bic=float(res.bic),
                                params=res.params, std_resid=np.asarray(res.std_resid))
            except Exception: continue
    return best

def _ljung_box_p(x, lags=10):
    try:
        from statsmodels.stats.diagnostic import acorr_ljungbox
        return float(acorr_ljungbox(x, lags=[lags], return_df=True)['lb_pvalue'].iloc[0])
    except Exception: return np.nan

def _engle_ng(z):
    z = np.asarray(z, float); z = z[np.isfinite(z)]
    if len(z) < 50: return np.nan
    z2 = z[1:]**2; zl = z[:-1]; Sn = (zl < 0).astype(float)
    X = np.column_stack([np.ones_like(zl), Sn, Sn*zl, (1-Sn)*zl])
    b, *_ = np.linalg.lstsq(X, z2, rcond=None); rss_f = float(((z2-X@b)**2).sum())
    rss_r = float(((z2-z2.mean())**2).sum()); n, k, qd = X.shape[0], X.shape[1], 3
    if rss_f <= 0: return np.nan
    F = ((rss_r-rss_f)/qd)/(rss_f/(n-k)); return float(1-_f_dist.cdf(F, qd, n-k))

def _garch_diag(std_resid):
    z = np.asarray(std_resid, float)
    return dict(lb_resid_p=_ljung_box_p(z, 10), lb_resid2_p=_ljung_box_p(z**2, 10),
                signbias_p=_engle_ng(z))

def _apply_garch_fixed(vs_full, vol, dist, params):
    """Causal conditional variance with parameters frozen at their formation estimates."""
    am = _arch_model(vs_full, mean='Zero', dist=dist, **_VOL_KW[vol])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); fixed = am.fix(np.asarray(params))
    cv = np.asarray(fixed.conditional_volatility, float)
    med = np.nanmedian(cv[cv > 0]) if np.any(cv > 0) else 1.0
    cv = np.where(np.isfinite(cv) & (cv > 0), cv, med); return cv**2

def _garch_persist(vol, p):
    a = float(p.get('alpha[1]', 0.0)); b = float(p.get('beta[1]', 0.0)); g = float(p.get('gamma[1]', 0.0))
    if vol == 'GARCH': return a+b
    if vol == 'GJR': return a+b+0.5*g
    return b

def _garch_gauss(vs_form, vs_full):
    def nll(pr):
        o, al, be = pr
        if o <= 0 or al < 0 or be < 0 or al+be >= 0.999: return 1e12
        s2 = np.empty(len(vs_form)); s2[0] = vs_form.var()
        for t in range(1, len(vs_form)): s2[t] = o+al*vs_form[t-1]**2+be*s2[t-1]
        return 0.5*np.sum(np.log(2*np.pi*s2)+vs_form**2/s2)
    r = _minimize(nll, [vs_form.var()*0.05, 0.05, 0.90], method='Nelder-Mead')
    o, al, be = r.x; s2 = np.empty(len(vs_full)); s2[0] = o/max(1-al-be, 1e-6)
    for t in range(1, len(vs_full)): s2[t] = o+al*vs_full[t-1]**2+be*s2[t-1]
    return s2, dict(vol_model='GARCH', dist='normal', bic=np.nan, g_omega=o, g_alpha=al,
                    g_gamma=np.nan, g_beta=be, g_nu=np.nan, g_lambda=np.nan,
                    garch_persist=al+be, engine='gauss',
                    lb_resid_p=np.nan, lb_resid2_p=np.nan, signbias_p=np.nan)

def model_pair_spread(y1, y2, f_end_idx, cfg):
    """KF + GARCH for one pair; returns series, KF hyperparameters and GARCH metadata."""
    psi = fit_kf(y1[:f_end_idx+1], y2[:f_end_idx+1], cfg)
    al, be, v, f = kf_filter(y1, y2, *psi, kappa=cfg.kf_kappa)
    vs_full = v*cfg.garch_scale; vs_form = vs_full[cfg.kf_burnin:f_end_idx+1]
    sel = select_garch(vs_form, cfg)
    if sel is not None:
        sigma2_S = _apply_garch_fixed(vs_full, sel['vol'], sel['dist'], sel['params'])/(cfg.garch_scale**2)
        p = sel['params']; persist = _garch_persist(sel['vol'], p)
        gmeta = dict(vol_model=sel['vol'], dist=sel['dist'], bic=sel['bic'],
                     g_omega=float(p.get('omega', np.nan)), g_alpha=float(p.get('alpha[1]', np.nan)),
                     g_gamma=float(p.get('gamma[1]', np.nan)), g_beta=float(p.get('beta[1]', np.nan)),
                     g_nu=float(p.get('nu', np.nan)), g_lambda=float(p.get('lambda', np.nan)),
                     garch_persist=persist, engine='arch', **_garch_diag(sel['std_resid']))
    else:
        s2_scaled, gmeta = _garch_gauss(vs_form, vs_full); sigma2_S = s2_scaled/(cfg.garch_scale**2)
        persist = gmeta['garch_persist']
    gmeta['igarch'] = bool(persist > 0.999)
    beta_dstd = float(np.std(np.diff(be))) if len(be) > 2 else np.nan
    gmeta['beta_dstd'] = beta_dstd
    gmeta['beta_iqr'] = float(np.subtract(*np.percentile(be, [75, 25]))) if len(be) else np.nan
    gmeta['tradable_hedge'] = bool(np.isfinite(beta_dstd) and beta_dstd <= cfg.beta_dstd_max)
    return al, be, v, f, sigma2_S, psi, gmeta

def model_spreads(cfg: Config, store: Store, pairs_df=None, force=False, verbose=True):
    """T2: model every screened pair -> spreads/series, spreads/meta."""
    if store.has('spreads/series') and not force:
        if verbose:
            meta = store.read('spreads/meta')
            print(f"[spreads] from cache: pairs={len(meta)} (force=False)")
        return store.read('spreads/series'), store.read('spreads/meta')
    if not _HAS_ARCH:
        print("[spreads] WARNING: `arch` not installed -> falling back to a Gaussian GARCH. Run `pip install arch`.")
    if pairs_df is None: pairs_df = store.read('selection/pairs')

    # --- guard: no pairs survived screening ---
    if pairs_df is None or len(pairs_df) == 0:
        if verbose: print("[spreads] no input pairs (empty screening) -- skipping.")
        empty_series, empty_meta = pd.DataFrame(), pd.DataFrame()
        store.write('spreads/series', empty_series, data_columns=['pair_id', 'phase'])
        store.write('spreads/meta', empty_meta, data_columns=['pair_id', 'group', 'vol_model', 'dist'])
        return empty_series, empty_meta

    tickers = pd.unique(pairs_df[['asset1', 'asset2']].values.ravel())
    close, _, _ = store.load_universe(tickers=tickers); logclose = np.log(close)
    fdays = int(cfg.formation_years*252); emb = cfg.embargo_days
    series_rows, meta_rows = [], []; n = len(pairs_df)
    for k, row in enumerate(pairs_df.itertuples(index=False), 1):
        a, b, win = row.asset1, row.asset2, row.window; pid = f"{a}_{b}_{win}"
        f0, t1 = pd.Timestamp(win.split('_')[0]), pd.Timestamp(win.split('_')[1])
        i0 = close.index.searchsorted(f0)
        if i0+fdays-1+emb >= len(close.index): continue
        f1 = close.index[i0+fdays-1]; e1 = close.index[i0+fdays-1+emb]
        df = pd.concat([logclose[a].loc[f0:t1], logclose[b].loc[f0:t1]], axis=1).dropna()
        if len(df) < fdays//2: continue
        dts = df.index; y1, y2 = df.iloc[:, 0].values, df.iloc[:, 1].values
        f_end_idx = int((dts <= f1).sum()-1)
        if f_end_idx < cfg.kf_burnin+60: continue
        try:
            al, be, v, f, s2S, psi, gmeta = model_pair_spread(y1, y2, f_end_idx, cfg)
        except Exception: continue
        phase = np.where(dts <= f1, 'formation', np.where(dts <= e1, 'embargo', 'trading'))
        series_rows.append(pd.DataFrame({'pair_id': pid, 'date': dts, 'phase': phase,
                                         'beta': be, 'spread': v, 'f': f, 'sigma2_S': s2S}))
        meta_rows.append(dict(pair_id=pid, asset1=a, asset2=b, window=win, group=row.group,
                              sig_eps2=psi[0], q_alpha=psi[1], q_beta=psi[2], snr=psi[2]/psi[0],
                              half_life=getattr(row, 'half_life', np.nan),
                              pvalue=getattr(row, 'pvalue', np.nan), hurst=getattr(row, 'hurst', np.nan),
                              vr=getattr(row, 'vr', np.nan),
                              f_start=str(f0.date()), f_end=str(f1.date()),
                              t_start=str(e1.date()), t_end=str(t1.date()),
                              n_obs=len(dts), n_trading=int((phase == 'trading').sum()), **gmeta))
        if verbose and (k % 20 == 0 or k == n):
            print(f"  [{k}/{n}] {pid} SNR={psi[2]/psi[0]:.0f} {gmeta['vol_model']}-{gmeta['dist']} "
                  f"persist={gmeta['garch_persist']:.3f}")
    series = pd.concat(series_rows, ignore_index=True) if series_rows else pd.DataFrame()
    meta = pd.DataFrame(meta_rows)
    store.write('spreads/series', series, data_columns=['pair_id', 'phase'])
    store.write('spreads/meta', meta, data_columns=['pair_id', 'group', 'vol_model', 'dist'])
    if verbose:
        if len(meta):
            print(f"[spreads] pairs={len(meta)} rows={len(series):,} | "
                  f"tradable={int(meta.tradable_hedge.sum())}/{len(meta)} | "
                  f"median SNR={meta.snr.median():.0f}")
        else:
            print("[spreads] no pair survived KF/GARCH modelling.")
    return series, meta

## 6 · Stage T3 — Regime detection (HMM)
A two-state Gaussian HMM is fitted with Baum–Welch on the formation window (6 random restarts, best log-likelihood). The observation is the **log realized volatility** of the spread changes (21-day window), standardized with formation moments. Using realized volatility rather than the z-score itself avoids a circularity between the signal and the filter that gates it.

The stable regime is the state with the lower mean volatility. The Hamilton filter gives the causal probability $\phi_t = \Pr(s_t = \text{stable} \mid y_{1:t})$ used for gating; the Viterbi path is stored only for diagnostics.

In [ ]:
def _gauss_B(y, mu, var, floor):
    var = np.maximum(var, floor)
    z = (y[:, None]-mu[None, :])**2/var[None, :]
    return np.exp(-0.5*z)/np.sqrt(2*np.pi*var[None, :])+1e-300

def _hmm_em(y, cfg, K=2, seed=0):
    """Baum-Welch (scaled forward-backward) for a K-state Gaussian HMM."""
    rng = np.random.default_rng(seed); fl = cfg.var_floor
    q = np.quantile(y, 0.5); lo, hi = y[y <= q], y[y > q]
    mu = np.array([lo.mean() if len(lo) else y.mean(), hi.mean() if len(hi) else y.mean()]) + rng.normal(0, 0.05, K)
    var = np.array([max(lo.var(), fl) if len(lo) else y.var(),
                    max(hi.var(), fl) if len(hi) else y.var()]) * rng.uniform(0.8, 1.2, K)
    A = np.array([[0.95, 0.05], [0.10, 0.90]]); pi = np.array([0.8, 0.2]); ll_old = -np.inf
    for _ in range(cfg.hmm_iters):
        B = _gauss_B(y, mu, var, fl); T = len(y)
        alpha = np.zeros((T, K)); c = np.zeros(T)
        alpha[0] = pi*B[0]; c[0] = alpha[0].sum()+1e-300; alpha[0] /= c[0]
        for t in range(1, T):
            alpha[t] = (alpha[t-1]@A)*B[t]; c[t] = alpha[t].sum()+1e-300; alpha[t] /= c[t]
        beta = np.zeros((T, K)); beta[-1] = 1.0
        for t in range(T-2, -1, -1): beta[t] = (A@(B[t+1]*beta[t+1]))/c[t+1]
        gamma = alpha*beta; gamma /= gamma.sum(1, keepdims=True)+1e-300
        xi = np.zeros((K, K))
        for t in range(T-1):
            m = (alpha[t][:, None]*A)*(B[t+1]*beta[t+1])[None, :]; xi += m/(m.sum()+1e-300)
        pi = gamma[0]; A = xi/(gamma[:-1].sum(0)[:, None]+1e-300); A /= A.sum(1, keepdims=True)
        g = gamma.sum(0)+1e-300; mu = (gamma*y[:, None]).sum(0)/g
        var = np.maximum((gamma*(y[:, None]-mu[None, :])**2).sum(0)/g, fl)
        ll = np.log(c).sum()
        if abs(ll-ll_old) < 1e-6: break
        ll_old = ll
    return dict(mu=mu, var=var, A=A, pi=pi, loglik=ll)

def _fit_hmm(y, cfg):
    best = None
    for s in range(cfg.hmm_restarts):
        try:
            m = _hmm_em(y, cfg, seed=cfg.seed + s)
            if np.isfinite(m['loglik']) and (best is None or m['loglik'] > best['loglik']): best = m
        except Exception: continue
    return best


def _hamilton(y, mu, var, A, pi, floor):
    """Causal filtered state probabilities (Hamilton filter)."""
    B = _gauss_B(y, mu, var, floor); T, K = B.shape
    alpha = np.zeros((T, K)); a = pi*B[0]; alpha[0] = a/(a.sum()+1e-300)
    for t in range(1, T):
        a = (alpha[t-1]@A)*B[t]; alpha[t] = a/(a.sum()+1e-300)
    return alpha

def _viterbi(y, mu, var, A, pi, floor):
    """Most likely state path (ex-post, diagnostics only)."""
    B = np.log(_gauss_B(y, mu, var, floor)); lA = np.log(A+1e-300); T, K = B.shape
    d = np.zeros((T, K)); psi = np.zeros((T, K), int); d[0] = np.log(pi+1e-300)+B[0]
    for t in range(1, T):
        for k in range(K):
            seq = d[t-1]+lA[:, k]; psi[t, k] = seq.argmax(); d[t, k] = seq.max()+B[t, k]
    path = np.zeros(T, int); path[-1] = d[-1].argmax()
    for t in range(T-2, -1, -1): path[t] = psi[t+1, path[t+1]]
    return path

def detect_regimes(cfg: Config, store: Store, only_tradable=True, force=False, verbose=True):
    """T3: fit the HMM per pair on formation data -> regimes/series, regimes/meta."""
    if store.has('regimes/series') and not force:
        if verbose:
            rm = store.read('regimes/meta')
            print(f"[regimes] from cache: pairs={len(rm)} (force=False)")
        return store.read('regimes/series'), store.read('regimes/meta')
    series = store.read('spreads/series'); meta = store.read('spreads/meta')

    # --- guard: no modelled spreads ---
    if series is None or len(series) == 0 or meta is None or len(meta) == 0:
        if verbose: print("[regimes] no input spreads -- skipping.")
        empty_regimes, empty_rmeta = pd.DataFrame(), pd.DataFrame()
        store.write('regimes/series', empty_regimes, data_columns=['pair_id', 'phase'])
        store.write('regimes/meta', empty_rmeta, data_columns=['pair_id'])
        return empty_regimes, empty_rmeta

    if only_tradable and 'tradable_hedge' in meta.columns:
        keep = set(meta.loc[meta.tradable_hedge, 'pair_id']); series = series[series.pair_id.isin(keep)]
    pair_ids = series.pair_id.unique()
    reg_rows, meta_rows = [], []
    for k, pid in enumerate(pair_ids, 1):
        d = series[series.pair_id == pid].sort_values('date')
        sp = d['spread'].values; dts = d['date'].values; ph = d['phase'].values
        if len(sp) < 100: continue
        dv = np.r_[0.0, np.diff(sp)]
        rv = pd.Series(dv).rolling(cfg.rv_window, min_periods=cfg.rv_window).std().values
        valid = np.isfinite(rv) & (rv > 0)
        if valid.sum() < 80: continue
        y_full = np.full(len(rv), np.nan); y_full[valid] = np.log(rv[valid])
        form = (ph == 'formation') & valid
        if form.sum() < 60: continue
        y = (y_full-y_full[form].mean())/(y_full[form].std()+1e-12)
        first = np.where(valid)[0][0]; y[:first] = y[first]; y = np.nan_to_num(y, nan=0.0)
        m = _fit_hmm(y[ph == 'formation'], cfg)
        if m is None: continue
        stable = int(np.argmin(m['mu']))
        phi = _hamilton(y, m['mu'], m['var'], m['A'], m['pi'], cfg.var_floor)[:, stable]
        vit = (_viterbi(y, m['mu'], m['var'], m['A'], m['pi'], cfg.var_floor) == stable).astype(int)
        reg_rows.append(pd.DataFrame({'pair_id': pid, 'date': dts, 'phase': ph,
                                      'phi': phi, 'stable_state': vit}))
        p00, p11 = m['A'][stable, stable], m['A'][1-stable, 1-stable]; tr = ph == 'trading'
        meta_rows.append(dict(pair_id=pid, stable_state=stable,
            mu_stable=float(m['mu'][stable]), mu_stress=float(m['mu'][1-stable]),
            p_stay_stable=float(p00), p_stay_stress=float(p11),
            dur_stable=1/(1-p00+1e-9), dur_stress=1/(1-p11+1e-9),
            phi_mean_trading=float(phi[tr].mean()) if tr.any() else np.nan,
            phi_std_trading=float(phi[tr].std()) if tr.any() else np.nan,
            regime_broken=int((phi[tr].mean() < 0.3)) if tr.any() else 0))
        if verbose and (k % 30 == 0 or k == len(pair_ids)):
            print(f"  [{k}/{len(pair_ids)}] {pid} dur_st={1/(1-p00+1e-9):.0f}d "
                  f"phi_tr={phi[tr].mean() if tr.any() else float('nan'):.2f}")
    regimes = pd.concat(reg_rows, ignore_index=True) if reg_rows else pd.DataFrame()
    rmeta = pd.DataFrame(meta_rows)
    store.write('regimes/series', regimes, data_columns=['pair_id', 'phase'])
    store.write('regimes/meta', rmeta, data_columns=['pair_id'])
    if verbose:
        if len(rmeta):
            print(f"[regimes] pairs={len(rmeta)} | stable duration={rmeta.dur_stable.median():.0f}d "
                  f"stress={rmeta.dur_stress.median():.0f}d | mean phi={rmeta.phi_mean_trading.median():.2f} "
                  f"| broken={rmeta.regime_broken.mean():.1%}")
        else:
            print("[regimes] no pair survived the HMM fit.")
    return regimes, rmeta

## 7 · Stage T4 — Signals
Two z-scores are computed for each pair and re-standardized causally (rolling 252-day mean and std, shifted by one day):

* `z_kf`  $= v_t / \sqrt{f_t + \hat\sigma^2_{S,t}}$, used by `kf` and `full`;
* `z_ols` $= (y_{1,t} - \hat\beta_{\text{OLS}}\, y_{2,t})$ with $\hat\beta_{\text{OLS}}$ estimated on formation, used by `ols`.

`gen_positions` is the trading state machine: enter when $|z_t| > z^*$ (and $\phi_t > \phi^*$ for `full`); exit when $|z_t| < z^{\text{exit}}$, on a stop at $|z_t| > z_{\text{stop}} = 4$, after $5 \times$ the half-life, or when $\phi_t$ falls below $\phi^*$.

In [ ]:
def causal_standardize(x, win, minp):
    """Rolling z-score using only past data (expanding window during warm-up)."""
    s = pd.Series(np.asarray(x, float))
    mu = s.rolling(win, min_periods=minp).mean().shift(1)
    sd = s.rolling(win, min_periods=minp).std().shift(1)
    mu_e = s.expanding(min_periods=20).mean().shift(1)
    sd_e = s.expanding(min_periods=20).std().shift(1)
    mu = mu.where(sd.notna(), mu_e); sd = sd.where(sd.notna(), sd_e)
    return ((s-mu)/(sd+1e-12)).fillna(0.0).values

def gen_positions(z, phi, z_star, z_exit, z_stop, phi_star, hl, mh):
    """Position path in {-1, 0, +1} (long spread = +1). phi=None disables the regime gate."""
    n = len(z); pos = np.zeros(n, int); st = 0; en = 0
    cap = mh*hl if (hl and np.isfinite(hl)) else np.inf
    for t in range(n):
        zt = z[t]; pt = phi[t] if phi is not None else 1.0
        if st == 0:
            if zt < -z_star and pt > phi_star: st = +1; en = t
            elif zt > z_star and pt > phi_star: st = -1; en = t
        else:
            held = t-en
            if abs(zt) < z_exit or abs(zt) > z_stop or held > cap: st = 0
            elif (phi is not None) and pt < phi_star: st = 0
        pos[t] = st
    return pos

def make_signals(cfg: Config, store: Store, force=False, verbose=True):
    """T4: trading-window z-scores, phi and hedge ratios -> signals/series."""
    if store.has('signals/series') and not force:
        if verbose:
            sg = store.read('signals/series')
            print(f"[signals] from cache: pairs={sg.pair_id.nunique() if len(sg) else 0} rows={len(sg):,} (force=False)")
        return store.read('signals/series')
    spreads = store.read('spreads/series'); smeta = store.read('spreads/meta')
    regimes = store.read('regimes/series'); pairs = store.read('selection/pairs')

    # --- guard: nothing upstream ---
    if pairs is None or len(pairs) == 0 or spreads is None or len(spreads) == 0:
        if verbose: print("[signals] no upstream input -- skipping.")
        empty = pd.DataFrame()
        store.write('signals/series', empty, data_columns=['pair_id'])
        return empty

    pairs = pairs.copy(); pairs['pair_id'] = pairs.asset1+'_'+pairs.asset2+'_'+pairs.window
    hl_map = pairs.set_index('pair_id')['half_life'].to_dict()
    tradable = set(smeta.loc[smeta.tradable_hedge, 'pair_id']) if 'tradable_hedge' in smeta else set(smeta.pair_id)
    tickers = pd.unique(pairs[['asset1', 'asset2']].values.ravel())
    close, _, _ = store.load_universe(tickers=tickers); logclose = np.log(close)
    phi_map = regimes.set_index(['pair_id', 'date'])['phi'] if len(regimes) else pd.Series(dtype=float)
    out_rows = []
    pids = [p for p in spreads.pair_id.unique() if p in tradable]
    for k, pid in enumerate(pids, 1):
        d = spreads[spreads.pair_id == pid].sort_values('date').reset_index(drop=True)
        rm = smeta[smeta.pair_id == pid].iloc[0]; a, b = rm['asset1'], rm['asset2']
        dts = d['date'].values; ph = d['phase'].values
        v = d['spread'].values; f = d['f'].values; s2 = d['sigma2_S'].values
        form = (ph == 'formation')
        if form.sum() < 60: continue
        z_kf = causal_standardize(v/np.sqrt(f+s2+1e-12), cfg.std_win, cfg.std_minp)
        ds = pd.DataFrame({'date': dts}).merge(
            pd.DataFrame({'date': logclose.index, a: logclose[a].values, b: logclose[b].values}),
            on='date', how='left')
        y1, y2 = ds[a].values, ds[b].values; ok = np.isfinite(y1) & np.isfinite(y2)
        beta_ols = np.polyfit(y2[ok & form], y1[ok & form], 1)[0]
        z_ols = causal_standardize(y1-beta_ols*y2, cfg.std_win, cfg.std_minp)
        phi = np.array([phi_map.get((pid, dt), np.nan) for dt in dts])
        phi = pd.Series(phi).ffill().fillna(1.0).values
        tr = (ph == 'trading'); idx = np.where(tr)[0]
        if len(idx) < 5: continue
        sl = slice(idx[0], idx[-1]+1)
        out_rows.append(pd.DataFrame({
            'pair_id': pid, 'date': dts[sl], 'z_ols': z_ols[sl], 'z_kf': z_kf[sl],
            'phi': phi[sl], 'beta_kf': d['beta'].values[sl], 'beta_ols': beta_ols}))
        if verbose and (k % 30 == 0 or k == len(pids)):
            print(f"  [{k}/{len(pids)}] {pid}")
    signals = pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame()
    store.write('signals/series', signals, data_columns=['pair_id'])
    if verbose:
        if len(signals):
            print(f"[signals] pairs={signals.pair_id.nunique()} rows={len(signals):,} | "
                  f"std z_kf={signals.z_kf.std():.2f} z_ols={signals.z_ols.std():.2f}")
        else:
            print("[signals] no pair survived.")
    return signals

## 8 · Stage T5 — Threshold calibration (CPCV, DSR, PBO)
For each specification, $(z^*, z^{\text{exit}})$ is chosen on the 5 × 4 grid in `Config` by maximizing the **net** Sharpe ratio (5 bps per side) of the equal-weight portfolio of all development pairs; $\phi^* = 0.65$ is fixed. Two overfitting diagnostics accompany the choice:

* **PBO** — probability of backtest overfitting from combinatorially symmetric cross-validation over 8 blocks;
* **DSR** — deflated Sharpe ratio, correcting for the 20 trials and for skewness and kurtosis of the returns.

P&L convention for a pair: $r_t = \tau_{t-1}\,[\Delta y_{1,t} - \beta_{t-1}\Delta y_{2,t}]$, with cost $c\,|\Delta\tau_t|\,(1 + |\beta_{t-1}|)$.

In [ ]:
from scipy.stats import norm as _norm, skew as _skew, kurtosis as _kurt

def _sharpe(r):
    r = np.asarray(r, float); s = r.std()
    return (r.mean()/s)*ANN if s > 1e-12 and len(r) > 5 else 0.0

def _deflated_sharpe(r, n_trials, sr_std):
    """Deflated Sharpe ratio (Bailey & Lopez de Prado). Returns (DSR, SR, SR0), annualized."""
    r = np.asarray(r, float); T = len(r); sd = r.std()
    if sd < 1e-12 or T < 10: return np.nan, 0.0, 0.0
    sr = r.mean()/sd; sk = float(_skew(r)); ku = float(_kurt(r, fisher=False)); g = 0.5772156649
    emc = (1-g)*_norm.ppf(1-1/n_trials)+g*_norm.ppf(1-1/(n_trials*np.e))
    sr0 = sr_std*emc; den = np.sqrt(max(1-sk*sr+((ku-1)/4)*sr**2, 1e-9))
    return float(_norm.cdf((sr-sr0)*np.sqrt(T-1)/den)), sr*ANN, sr0*ANN

def _cscv_pbo(R, S):
    """Probability of backtest overfitting via CSCV. R: (n_configs, T) returns, S: blocks."""
    N, T = R.shape
    if T < S*5: return np.nan
    bsz = T//S; blocks = [np.arange(i*bsz, (i+1)*bsz if i < S-1 else T) for i in range(S)]
    lams = []
    for IS in itertools.combinations(range(S), S//2):
        OOS = [b for b in range(S) if b not in IS]
        ii = np.concatenate([blocks[b] for b in IS]); oo = np.concatenate([blocks[b] for b in OOS])
        sr_is = np.array([_sharpe(R[n, ii]) for n in range(N)])
        sr_oos = np.array([_sharpe(R[n, oo]) for n in range(N)])
        ns = int(np.argmax(sr_is)); rank = (sr_oos.argsort().argsort()[ns]+1)/(N+1)
        rank = min(max(rank, 1e-6), 1-1e-6); lams.append(np.log(rank/(1-rank)))
    return float((np.array(lams) <= 0).mean())

def _build_pair_panel(cfg, store):
    """Per-pair arrays (leg log-prices, betas, z-scores, phi) shared by T5 and the backtests."""
    sig = store.read('signals/series'); spreads = store.read('spreads/series')
    smeta = store.read('spreads/meta'); pairs = store.read('selection/pairs')

    # --- guard: no pairs in any upstream input ---
    if (pairs is None or len(pairs) == 0 or 'asset1' not in pairs.columns
            or sig is None or len(sig) == 0):
        return {}, np.array([], dtype='datetime64[ns]'), {}

    pairs = pairs.copy(); pairs['pair_id'] = pairs.asset1+'_'+pairs.asset2+'_'+pairs.window
    hl_map = pairs.set_index('pair_id')['half_life'].to_dict()
    tickers = pd.unique(pairs[['asset1', 'asset2']].values.ravel())
    close, _, _ = store.load_universe(tickers=tickers); lc = np.log(close)
    beta_ols_map = sig.groupby('pair_id')['beta_ols'].first().to_dict()
    P = {}
    for pid in sig.pair_id.unique():
        rm = smeta[smeta.pair_id == pid].iloc[0]; a, b = rm['asset1'], rm['asset2']
        g = sig[sig.pair_id == pid].sort_values('date')
        gb = g.merge(spreads[['pair_id', 'date', 'beta']], on=['pair_id', 'date'], how='left')
        df = pd.DataFrame({'date': g.date.values}).merge(
            pd.DataFrame({'date': lc.index, 'y1': lc[a].values, 'y2': lc[b].values}),
            on='date', how='left')
        P[pid] = dict(date=g.date.values, z_kf=g.z_kf.values, z_ols=g.z_ols.values,
                      phi=g.phi.values, beta_kf=gb.beta.values, beta_ols=beta_ols_map.get(pid, np.nan),
                      y1=df.y1.values, y2=df.y2.values, hl=hl_map.get(pid, np.nan))
    all_dates = (np.array(sorted(set(np.concatenate([p['date'] for p in P.values()]))))
                 if P else np.array([], dtype='datetime64[ns]'))
    return P, all_dates, {d: i for i, d in enumerate(all_dates)}


def _pair_net_returns(p, spec, zs, ze, cfg, c):
    """Net daily returns of one pair for a given spec and thresholds. Returns (net, pos)."""
    z = p['z_kf'] if spec != 'ols' else p['z_ols']
    phi = p['phi'] if spec == 'full' else None
    beta = p['beta_kf'] if spec != 'ols' else np.full(len(z), p['beta_ols'])
    ps = cfg.phi_star if spec == 'full' else -1.0
    pos = gen_positions(z, phi, zs, ze, cfg.z_stop, ps, p['hl'], cfg.max_hold_mult)
    y1, y2 = p['y1'], p['y2']
    base = np.zeros(len(z)); base[1:] = (y1[1:]-y1[:-1])-beta[:-1]*(y2[1:]-y2[:-1])
    gross = np.r_[0.0, pos[:-1]*base[1:]]
    turn = np.r_[0.0, np.abs(np.diff(pos))*(1+np.abs(beta[:-1]))]
    return gross - c*turn, pos

def _portfolio_net(P, all_dates, dpos, spec, zs, ze, cfg, c, subset=None):
    """Average net return across pairs with an open position (used for calibration)."""
    S = np.zeros(len(all_dates)); Cnt = np.zeros(len(all_dates))
    items = P.items() if subset is None else [(k, P[k]) for k in subset if k in P]
    for pid, p in items:
        net, pos = _pair_net_returns(p, spec, zs, ze, cfg, c)
        ii = np.array([dpos[d] for d in p['date']]); S[ii] += net; Cnt[ii] += (np.r_[0, pos[:-1]] != 0)
    return S/np.maximum(Cnt, 1)

def optimize_global(cfg: Config, store: Store, force=False, verbose=True):
    """T5: grid search of (z*, z_exit) per spec on development data -> params/thresholds."""
    if store.has('params/thresholds') and not force:
        thr = store.read('params/thresholds')
        if verbose: print(f"[optimize] from cache:\n{thr.to_string(index=False)}")
        return thr
    P, all_dates, dpos = _build_pair_panel(cfg, store); c = cfg.cost_bps/1e4
    results = {}
    for spec in ['ols', 'kf', 'full']:
        cands = [(zs, ze) for zs in cfg.grid_zstar for ze in cfg.grid_zexit]
        R = np.array([_portfolio_net(P, all_dates, dpos, spec, zs, ze, cfg, c) for (zs, ze) in cands])
        srs = np.array([_sharpe(R[i]) for i in range(len(cands))])
        best = int(np.argmax(srs)); zs, ze = cands[best]
        pbo = _cscv_pbo(R, cfg.cscv_blocks)
        dsr, sr_a, sr0_a = _deflated_sharpe(R[best], len(cands), (srs/ANN).std())
        results[spec] = dict(z_star=zs, z_exit=ze,
                             phi_star=(cfg.phi_star if spec == 'full' else np.nan),
                             z_stop=cfg.z_stop, max_hold_mult=cfg.max_hold_mult, cost_bps=cfg.cost_bps,
                             sharpe_net_dev=float(srs[best]), pbo=pbo, dsr=dsr,
                             sr_deflated_benchmark=sr0_a, n_trials=len(cands))
        if verbose:
            print(f"[{spec.upper():4s}] z*={zs} z_exit={ze} "
                  f"phi*={cfg.phi_star if spec=='full' else '—'} | "
                  f"SR_net={srs[best]:.2f} PBO={pbo:.2f} DSR={dsr:.2f}")
    thr = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'spec'})
    store.write('params/thresholds', thr)
    if verbose: print("[optimize] params/thresholds saved (optimized on net returns).")
    return thr

## 9 · Quality ranking and backtest
**Quality score.** Pairs that pass the executability filter are ranked by the sum of cross-sectional z-scores of seven components: cointegration strength ($-\log p$), distance of the half-life from the 25-day target, Hurst exponent, variance ratio, hedge-ratio stability $\mathrm{std}(\Delta\beta)$, GARCH adequacy (mean of the diagnostic p-values) and a regime term. The top-$N$ pairs form the portfolio.

> **Caveat.** The regime term uses `phi_mean_trading` and `regime_broken`, i.e. the average filtered probability over the *trading* window. $\phi_t$ itself is causal, but its trading-window average is not known at the start of the window, so this component introduces a mild look-ahead into the ranking (not into the positions). It affects which pairs enter the top-$N$ in the same way for all three specifications.

**Two portfolio views.**

* `backtest` — *flat / fixed-capital view*: capital $1/N$ per pair over the whole sample. It isolates per-position risk, but exposure is diluted because most pairs are only active in their own window.
* `backtest_invested` — *invested view*: in each window the capital is split among the pairs with an open position that day and the window returns are chained into one equity curve. This is closer to what a fund re-deploying its capital every year would experience.

`backtest_by_window` breaks the flat view down by trading window.

In [ ]:
def _zsc(s):
    s = pd.to_numeric(s, errors='coerce'); return (s-s.mean())/(s.std()+1e-12)

def quality_score(cfg: Config, store: Store, force=False, verbose=True):
    """Rank tradable pairs by a composite of standardized quality components -> selection/quality."""
    if store.has('selection/quality') and not force:
        q = store.read('selection/quality')
        if verbose: print(f"[quality] from cache: {len(q)} pairs")
        return q
    smeta = store.read('spreads/meta')

    # --- guard: no spread metadata ---
    if smeta is None or len(smeta) == 0:
        if verbose: print("[quality] no input pairs -- skipping.")
        empty = pd.DataFrame()
        store.write('selection/quality', empty, data_columns=['pair_id'])
        return empty

    rmeta = store.read('regimes/meta') if store.has('regimes/meta') else None
    df = smeta.copy()
    if 'tradable_hedge' in df.columns: df = df[df.tradable_hedge].copy()
    if rmeta is not None and len(rmeta):
        df = df.merge(rmeta[['pair_id', 'phi_mean_trading', 'regime_broken']], on='pair_id', how='left')
    if len(df) == 0:
        if verbose: print("[quality] no tradable pairs -- skipping.")
        empty = pd.DataFrame()
        store.write('selection/quality', empty, data_columns=['pair_id'])
        return empty
    comp = pd.DataFrame(index=df.index)
    comp['coint']      = _zsc(-np.log(df['pvalue'].clip(1e-8, 1)))
    comp['halflife']   = _zsc(-(np.log(df['half_life']/cfg.half_life_target)).abs())
    comp['hurst']      = _zsc(-df['hurst'])
    comp['vr']         = _zsc(-df['vr'])
    comp['hedge_exec'] = _zsc(-df['beta_dstd'])
    comp['garch_adeq'] = _zsc(df[['lb_resid_p', 'lb_resid2_p', 'signbias_p']].mean(axis=1))
    if 'phi_mean_trading' in df.columns:
        comp['regime'] = _zsc(df['phi_mean_trading'].fillna(df['phi_mean_trading'].median())) \
                         - 2.0*df['regime_broken'].fillna(0)
    else:
        comp['regime'] = 0.0
    df['quality_score'] = comp.sum(axis=1)
    for k in comp.columns: df[f'q_{k}'] = comp[k]
    df = df.sort_values('quality_score', ascending=False).reset_index(drop=True)
    df['rank'] = np.arange(1, len(df)+1)
    keep = ['pair_id', 'asset1', 'asset2', 'window', 'group', 'quality_score', 'rank',
            'pvalue', 'half_life', 'hurst', 'vr', 'beta_dstd', 'vol_model', 'dist',
            'snr'] + [f'q_{k}' for k in comp.columns]
    if 'phi_mean_trading' in df.columns: keep += ['phi_mean_trading', 'regime_broken']
    store.write('selection/quality', df[keep], data_columns=['pair_id'])
    if verbose:
        print(f"[quality] {len(df)} pairs. TOP {cfg.top_n}:")
        print(df[['rank', 'pair_id', 'quality_score', 'half_life', 'hurst', 'beta_dstd']]
              .head(cfg.top_n).to_string(index=False))
    return df


def _metrics(net, pos, cost):
    """Performance metrics of a daily net-return series."""
    net = np.asarray(net); eq = np.cumsum(net); dd = eq-np.maximum.accumulate(eq)
    sd = net.std()+1e-12; ntr = int((np.abs(np.diff(pos)) > 0).sum())//2; maxdd = dd.min()
    return dict(sharpe=net.mean()/sd*ANN, ann_ret=net.mean()*252, total=eq[-1], maxDD=maxdd,
                calmar=(net.mean()*252)/abs(maxdd) if maxdd < -1e-9 else np.nan,
                cvar95=net[net <= np.percentile(net, 5)].mean() if (net < 0).any() else 0.0,
                n_trades=ntr, exposure=(pos != 0).mean(),
                hit=(net[net != 0] > 0).mean() if (net != 0).any() else np.nan, cost_tot=cost.sum())

def backtest(cfg: Config, store: Store, top_n=None, per_pair=True, plot=True, verbose=True):
    """Flat view: per-pair results for the top-N pairs and the fixed-capital (1/N) portfolio."""
    top_n = top_n or cfg.top_n
    thr = store.read('params/thresholds').set_index('spec')
    qual = store.read('selection/quality')

    # --- guard: nothing to backtest ---
    if qual is None or len(qual) == 0:
        if verbose: print("[backtest] no pairs available (empty quality ranking).")
        return {}, pd.DataFrame()

    P, all_dates, dpos = _build_pair_panel(cfg, store); c = cfg.cost_bps/1e4
    if not P:
        if verbose: print("[backtest] no pairs available (empty panel).")
        return {}, pd.DataFrame()
    # keep only the top-N pairs present in the panel (_build_pair_panel may
    # drop pairs with insufficient data)
    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    if verbose and len(top_ids) < len(ranked):
        print(f"[backtest] {len(ranked)-len(top_ids)} top-{top_n} pairs dropped (insufficient data); "
              f"using {len(top_ids)} pairs.")
    if not top_ids:
        print("[backtest] no pairs available.")
        return {}, pd.DataFrame()

    # --- per pair ---
    per_pair_res = {}
    if per_pair:
        try:
            import matplotlib.pyplot as plt
            have_plt = True
        except Exception:
            have_plt = False; plot = False
        for pid in top_ids:
            p = P[pid]; rows = {}; curves = {}
            for spec in ['ols', 'kf', 'full']:
                zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
                net, pos = _pair_net_returns(p, spec, zs, ze, cfg, c)
                rows[spec] = _metrics(net, pos, c*np.r_[0, np.abs(np.diff(pos))*(1+np.abs(
                    (p['beta_kf'] if spec != 'ols' else np.full(len(pos), p['beta_ols']))[:-1]))])
                curves[spec] = (pd.to_datetime(p['date']), np.cumsum(net), pos)
            res = pd.DataFrame(rows).T; per_pair_res[pid] = res
            if verbose:
                print(f"\n{'='*64}\n {pid}\n{'='*64}")
                print(res[['sharpe', 'ann_ret', 'maxDD', 'calmar', 'cvar95', 'n_trades', 'exposure', 'hit']].round(3).to_string())
            if plot and have_plt:
                fig, ax = plt.subplots(figsize=(10, 3.5))
                _C = globals().get('C', {})
                col = {'ols': _C.get('normal', '#444'), 'kf': _C.get('bull', '#1f77b4'),
                       'full': _C.get('gfc', '#d62728')}
                lab = {'ols': 'OLS', 'kf': 'KF+GARCH', 'full': 'KF+GARCH+HMM'}
                ls = {'ols': '--', 'kf': '-', 'full': '-'}
                for s in ['ols', 'kf', 'full']:
                    dts, eq, _ = curves[s]
                    ax.plot(dts, eq, color=col[s], lw=1.6, ls=ls[s], label=lab[s])
                ax.set_title(f'{pid}  —  net cumulative P&L', fontweight='bold')
                ax.set_ylabel('cumulative net P&L')
                ax.legend(frameon=False); ax.axhline(0, color='k', lw=.6)
                plt.tight_layout(); plt.show()

    # --- top-N portfolio: fixed capital (1/N per pair) ---
    N = len(top_ids)
    agg = {}
    for spec in ['ols', 'kf', 'full']:
        zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
        S = np.zeros(len(all_dates)); POS = np.zeros(len(all_dates))
        for pid in top_ids:
            if pid not in P: continue
            net, pos = _pair_net_returns(P[pid], spec, zs, ze, cfg, c)
            ii = np.array([dpos[d] for d in P[pid]['date']])
            S[ii] += net
            POS[ii] += (pos != 0)
        r = S / max(N, 1)
        eq = np.cumsum(r); dd = eq - np.maximum.accumulate(eq)
        agg[spec] = dict(sharpe=_sharpe(r), ann_ret=r.mean()*252, maxDD=dd.min(),
                         calmar=(r.mean()*252)/abs(dd.min()) if dd.min() < -1e-9 else np.nan,
                         cvar95=r[r <= np.percentile(r, 5)].mean() if (r < 0).any() else 0.0,
                         exposure=(POS/max(N, 1)).mean())
    aggdf = pd.DataFrame(agg).T
    if verbose:
        print(f"\n{'='*64}\n TOP-{top_n} PORTFOLIO (fixed capital 1/N)\n{'='*64}")
        print(aggdf[['sharpe', 'ann_ret', 'maxDD', 'calmar', 'cvar95', 'exposure']].round(3).to_string())
    return per_pair_res, aggdf

In [ ]:
def backtest_by_window(cfg: Config, store: Store, top_n=None, verbose=True):
    """Flat-view Sharpe and max drawdown for each trading window separately."""
    top_n = top_n or cfg.top_n
    thr = store.read('params/thresholds').set_index('spec')
    qual = store.read('selection/quality')
    top_ids = set(qual.nsmallest(top_n, 'rank')['pair_id'])
    P, _, _ = _build_pair_panel(cfg, store); c = cfg.cost_bps/1e4
    from collections import defaultdict
    by_win = defaultdict(list)
    for pid in top_ids:
        if pid in P:
            by_win['_'.join(pid.split('_')[-2:])].append(pid)   # window id = <formation start>_<trading end>
    rows = []
    for win in sorted(by_win, key=lambda w: w.split('_')[-1]):
        pids = by_win[win]; N = len(pids)
        wdates = np.array(sorted(set(np.concatenate([P[pid]['date'] for pid in pids]))))
        wpos = {d: i for i, d in enumerate(wdates)}
        t_end = win.split('_')[-1]
        rec = {'trading_end': t_end, 'n_pairs': N}
        for spec in ['ols', 'kf', 'full']:
            zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
            S = np.zeros(len(wdates))
            for pid in pids:
                net, _ = _pair_net_returns(P[pid], spec, zs, ze, cfg, c)
                ii = np.array([wpos[d] for d in P[pid]['date']]); S[ii] += net
            r = S/max(N, 1); eq = np.cumsum(r); dd = eq-np.maximum.accumulate(eq)
            rec[f'{spec}_SR'] = round(_sharpe(r), 3)
            rec[f'{spec}_maxDD'] = round(float(dd.min()), 4)
        rows.append(rec)
    df = pd.DataFrame(rows)
    if verbose:
        print(f"\n{'='*78}\n COMPARISON BY TRADING WINDOW (top-{top_n}, fixed capital within each window)\n{'='*78}")
        print(df.to_string(index=False))
        # summary: in how many windows each spec has the shallowest maxDD (ties go to the first column)
        best_dd = df[['ols_maxDD', 'kf_maxDD', 'full_maxDD']].idxmax(axis=1)
        print("\nShallowest maxDD, number of windows:",
              {k.replace('_maxDD', ''): int((best_dd == k).sum()) for k in ['ols_maxDD', 'kf_maxDD', 'full_maxDD']})
    return df

In [ ]:
def backtest_invested(cfg: Config, store: Store, top_n=None, verbose=True, plot=False):
    """Invested view: per-window allocation to active pairs, window returns chained.

    Unlike the fixed-capital view, capital is not diluted over pairs that are not
    trading. Thresholds are the frozen ones in params/thresholds."""
    top_n = top_n or cfg.top_n
    thr = store.read('params/thresholds').set_index('spec')
    qual = store.read('selection/quality')
    P, _, _ = _build_pair_panel(cfg, store); c = cfg.cost_bps/1e4
    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    # group pairs by window (<formation start>_<trading end>)
    from collections import defaultdict
    by_win = defaultdict(list)
    for pid in top_ids:
        by_win['_'.join(pid.split('_')[-2:])].append(pid)
    wins = sorted(by_win, key=lambda w: w.split('_')[-1])

    out = {}; curves = {}
    for spec in ['ols', 'kf', 'full']:
        zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
        seg_returns = []; seg_dates = []
        for win in wins:
            pids = by_win[win]
            wdates = np.array(sorted(set(np.concatenate([P[pid]['date'] for pid in pids]))))
            wpos = {d: i for i, d in enumerate(wdates)}
            S = np.zeros(len(wdates)); ACT = np.zeros(len(wdates))
            for pid in pids:
                net, pos = _pair_net_returns(P[pid], spec, zs, ze, cfg, c)
                ii = np.array([wpos[d] for d in P[pid]['date']])
                S[ii] += net; ACT[ii] += (np.r_[0, pos[:-1]] != 0)
            # capital split among the pairs with an open position that day;
            # with no active pair the return is 0 (cash).
            r = np.where(ACT > 0, S/np.maximum(ACT, 1), 0.0)
            seg_returns.append(r); seg_dates.append(wdates)
        r = np.concatenate(seg_returns); dts = pd.to_datetime(np.concatenate(seg_dates))
        order = np.argsort(dts.values); r = r[order]; dts = dts[order]
        eq = np.cumsum(r); dd = eq - np.maximum.accumulate(eq)
        out[spec] = dict(sharpe=_sharpe(r), ann_ret=r.mean()*252, total=eq[-1], maxDD=dd.min(),
                         calmar=(r.mean()*252)/abs(dd.min()) if dd.min() < -1e-9 else np.nan,
                         cvar95=r[r <= np.percentile(r, 5)].mean() if (r < 0).any() else 0.0,
                         frac_active_days=float((r != 0).mean()))
        curves[spec] = (dts, eq)
    res = pd.DataFrame(out).T
    if verbose:
        print(f"\n{'='*70}\n INVESTED BACKTEST — top-{top_n}, per-window allocation, chained equity\n{'='*70}")
        print(res[['sharpe', 'ann_ret', 'total', 'maxDD', 'calmar', 'cvar95', 'frac_active_days']].round(3).to_string())
    if plot:
        try:
            import matplotlib.pyplot as plt
            _C = globals().get('C', {})
            col = {'ols': _C.get('normal', '#444'), 'kf': _C.get('bull', '#1f77b4'), 'full': _C.get('gfc', '#d62728')}
            lab = {'ols': 'OLS', 'kf': 'KF+GARCH', 'full': 'KF+GARCH+HMM'}
            ls = {'ols': '--', 'kf': '-', 'full': '-'}
            fig, ax = plt.subplots(figsize=(11, 4))
            for s in ['ols', 'kf', 'full']:
                dts, eq = curves[s]; ax.plot(dts, eq, color=col[s], lw=1.6, ls=ls[s], label=lab[s])
            ax.set_title(f'Chained equity (invested view, top-{top_n})', fontweight='bold')
            ax.set_ylabel('cumulative net P&L'); ax.legend(frameon=False); ax.axhline(0, color='k', lw=.6)
            plt.tight_layout()
            if 'OUTPUT_DIR' in globals(): plt.savefig(globals()['OUTPUT_DIR'] + f'equity_invested_top{top_n}.png')
            plt.show()
        except Exception:
            pass
    return res, curves

## 10 · Orchestrators
* `run_pipeline` — development chain T1 → T5 → quality → backtest. `force_from` recomputes from a given stage onward.
* `preview_holdout_windows` — prints the holdout windows before running them (no-look-ahead check).
* `run_holdout` — holdout chain in the `holdout` namespace. The development thresholds are copied and frozen; `optimize_global` is **not** called.

### 10b · Second independent holdout: the 2008–2009 crisis
The main development sample (up to 2019) contains the 2008–2009 crisis, so its thresholds cannot be tested on that period. `run_crisis_holdout_2008` repeats the whole protocol with a development sample ending on 2007-06-30 (namespace `crisis2008_dev`) and a holdout bounded to 2008-01-01 → 2009-12-31 (namespace `crisis2008_holdout`). `compare_two_crises` puts the four net Sharpe columns side by side.

In [ ]:
def run_pipeline(cfg: Config, force_from=None, top_n=None):
    """Development run. force_from in {None,'screen','spreads','regimes','signals','optimize','quality'}."""
    order = ['screen', 'spreads', 'regimes', 'signals', 'optimize', 'quality']
    fi = order.index(force_from) if force_from in order else len(order)
    store = Store(cfg)
    pairs_df = screen(cfg, store, force=(fi <= 0))
    model_spreads(cfg, store, pairs_df, force=(fi <= 1))
    detect_regimes(cfg, store, force=(fi <= 2))
    make_signals(cfg, store, force=(fi <= 3))
    optimize_global(cfg, store, force=(fi <= 4))
    quality_score(cfg, store, force=(fi <= 5))
    return backtest(cfg, store, top_n=top_n)


def preview_holdout_windows(cfg: Config):
    """Print the holdout windows (sanity check before running)."""
    store = Store(cfg); close, _, _ = store.load_universe()
    wins = list(walk_forward(close.index, cfg, holdout=True))
    print(f"HOLDOUT windows (trading end >= {cfg.holdout_start}): {len(wins)}")
    for (f0, f1, e1, t1) in wins:
        print(f"  formation {f0.date()}–{f1.date()} | embargo→ {e1.date()} | trading →{t1.date()}")
    return wins

def run_holdout(cfg: Config, force=False, top_n=None, verbose=True):
    """Holdout run with thresholds frozen from development. Backtested once."""
    dev = Store(cfg)                      # development namespace (thresholds)
    hold = Store(cfg, ns='holdout')       # holdout namespace (isolated outputs)

    if not dev.has('params/thresholds'):
        raise RuntimeError("params/thresholds (dev) missing: run run_pipeline first (T5).")
    thr_dev = dev.read('params/thresholds')
    hold.write('params/thresholds', thr_dev)   # freeze the development thresholds in the holdout namespace
    if verbose:
        print("=== HOLDOUT 2020+ | thresholds FROZEN from development ===")
        print(thr_dev[['spec', 'z_star', 'z_exit', 'phi_star']].to_string(index=False), '\n')

    pairs_h = screen(cfg, hold, force=force, holdout=True, verbose=verbose)
    model_spreads(cfg, hold, pairs_h, force=force, verbose=verbose)
    detect_regimes(cfg, hold, force=force, verbose=verbose)
    make_signals(cfg, hold, force=force, verbose=verbose)
    quality_score(cfg, hold, force=force, verbose=verbose)   # ranking on the holdout windows
    # NB: no optimize_global here; thresholds are the development ones.
    return backtest(cfg, hold, top_n=top_n, verbose=verbose)

In [ ]:
from dataclasses import replace


def run_pipeline_ns(cfg: Config, ns=None, force_from=None, top_n=None):
    """run_pipeline() with an explicit namespace, so the main results (ns=None) are never overwritten."""
    order = ['screen', 'spreads', 'regimes', 'signals', 'optimize', 'quality']
    fi = order.index(force_from) if force_from in order else len(order)
    store = Store(cfg, ns=ns)
    pairs_df = screen(cfg, store, force=(fi <= 0))
    model_spreads(cfg, store, pairs_df, force=(fi <= 1))
    detect_regimes(cfg, store, force=(fi <= 2))
    make_signals(cfg, store, force=(fi <= 3))
    optimize_global(cfg, store, force=(fi <= 4))
    quality_score(cfg, store, force=(fi <= 5))
    return backtest(cfg, store, top_n=top_n)


def run_holdout_ns(cfg: Config, dev_ns=None, hold_ns='holdout', force=False, top_n=None, verbose=True):
    """run_holdout() with explicit namespaces for the development (threshold source) and the holdout."""
    dev = Store(cfg, ns=dev_ns)
    hold = Store(cfg, ns=hold_ns)
    if not dev.has('params/thresholds'):
        raise RuntimeError(f"params/thresholds missing in namespace '{dev_ns}': "
                            f"run run_pipeline_ns(cfg, ns='{dev_ns}') first.")
    thr_dev = dev.read('params/thresholds')
    hold.write('params/thresholds', thr_dev)
    if verbose:
        print(f"=== HOLDOUT (ns='{hold_ns}') | thresholds FROZEN from ns='{dev_ns}' ===")
        print(thr_dev[['spec', 'z_star', 'z_exit', 'phi_star']].to_string(index=False), '\n')

    pairs_h = screen(cfg, hold, force=force, holdout=True, verbose=verbose)
    model_spreads(cfg, hold, pairs_h, force=force, verbose=verbose)
    detect_regimes(cfg, hold, force=force, verbose=verbose)
    make_signals(cfg, hold, force=force, verbose=verbose)
    quality_score(cfg, hold, force=force, verbose=verbose)
    return backtest(cfg, hold, top_n=top_n, verbose=verbose)


def run_crisis_holdout_2008(base_cfg: Config, pre_crisis_dev_end='2007-06-30',
                             crisis_start='2008-01-01', crisis_end='2009-12-31',
                             top_n=None, verbose=True):
    """End-to-end 2008-2009 experiment. Returns (agg_pre_crisis_dev, agg_crisis_holdout),
    with the same structure as (agg_dev, agg_h) of the main run."""
    top_n = top_n or base_cfg.top_n
    cfg_crisis = replace(base_cfg, dev_end=pre_crisis_dev_end,
                         holdout_start=crisis_start, holdout_end=crisis_end)

    if verbose:
        print(f"=== 2008-2009 CRISIS EXPERIMENT ===")
        print(f"Development (threshold optimization): data up to {pre_crisis_dev_end}")
        print(f"Holdout (frozen thresholds, single test): {crisis_start} -> {crisis_end}\n")

    _, agg_pre_crisis_dev = run_pipeline_ns(cfg_crisis, ns='crisis2008_dev', top_n=top_n)
    if verbose:
        print(f"\n=== PRE-CRISIS DEVELOPMENT (up to {pre_crisis_dev_end}) ===")
        print(agg_pre_crisis_dev.round(3).to_string())

    _, agg_crisis_hold = run_holdout_ns(cfg_crisis, dev_ns='crisis2008_dev',
                                        hold_ns='crisis2008_holdout', force=True, top_n=top_n)
    if verbose:
        print(f"\n=== CRISIS HOLDOUT ({crisis_start} -> {crisis_end}) ===")
        print(agg_crisis_hold.round(3).to_string())

    return agg_pre_crisis_dev, agg_crisis_hold


def compare_two_crises(agg_dev, agg_h, agg_pre_crisis_dev, agg_crisis_hold):
    """Four-column table of net Sharpe ratios: both experiments side by side."""
    specs = ['ols', 'kf', 'full']

    def _col(df, label):
        # guard: an empty backtest (e.g. no pair found in a short crisis
        # window) returns a NaN column instead of raising a KeyError.
        if df is None or len(df) == 0 or 'sharpe' not in df.columns:
            print(f"[compare_two_crises] WARNING: '{label}' is empty (no pair "
                  f"available in that window) -- column filled with NaN.")
            return pd.Series(np.nan, index=specs)
        return df.reindex(specs)['sharpe']

    out = pd.DataFrame({
        ('2008-2009', 'dev_pre_crisis'): _col(agg_pre_crisis_dev, 'agg_pre_crisis_dev'),
        ('2008-2009', 'crisis_holdout'): _col(agg_crisis_hold, 'agg_crisis_hold'),
        ('2020-2026', 'dev'):            _col(agg_dev, 'agg_dev'),
        ('2020-2026', 'holdout'):        _col(agg_h, 'agg_h'),
    })
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    print(f"\n{'='*70}\n TWO INDEPENDENT HOLDOUTS — net Sharpe by specification\n{'='*70}")
    print(out.round(3).to_string())
    return out

## 11 · Robustness and diagnostic tools
These functions only read the cached panels; none of them re-optimizes the thresholds.

### 11.1 · Bootstrap confidence intervals for the Sharpe ratio
With a few dozen pairs in the holdout, point Sharpe ratios hide a large standard error. A moving-block bootstrap (Künsch, 1989; blocks of 20 trading days) preserves the serial dependence created by multi-day positions. Intervals are computed for both portfolio views.

In [ ]:
def _portfolio_returns_fixed(cfg: Config, store: Store, top_n=None):
    """Daily returns of the fixed-capital (1/N) portfolio for each spec.
    Same aggregation as backtest() (S/N), but returns the series itself."""
    top_n = top_n or cfg.top_n
    thr = store.read('params/thresholds').set_index('spec')
    qual = store.read('selection/quality')
    P, all_dates, dpos = _build_pair_panel(cfg, store); c = cfg.cost_bps/1e4
    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    N = len(top_ids)
    out = {}
    for spec in ['ols', 'kf', 'full']:
        zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
        S = np.zeros(len(all_dates))
        for pid in top_ids:
            net, pos = _pair_net_returns(P[pid], spec, zs, ze, cfg, c)
            ii = np.array([dpos[d] for d in P[pid]['date']])
            S[ii] += net
        out[spec] = S / max(N, 1)
    return out, all_dates


def _returns_from_invested_curves(curves):
    """Recover daily returns from the equity curves of backtest_invested() (r = diff of cumsum)."""
    out = {}
    for spec, (dts, eq) in curves.items():
        eq = np.asarray(eq, float)
        r = np.diff(np.r_[0.0, eq])
        out[spec] = r
    return out


def block_bootstrap_sharpe_ci(r, n_boot=2000, block=20, ci=0.90, seed=0):
    """Moving-block bootstrap of the annualized Sharpe ratio. `block` is in trading
    days (20 ~ one month). Returns (sharpe_hat, lo, hi)."""
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    n = len(r)
    if n < block * 3:
        return _sharpe(r), np.nan, np.nan
    n_blocks_needed = int(np.ceil(n / block))
    rng = np.random.default_rng(seed)
    sharpes = np.empty(n_boot)
    max_start = n - block
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([r[s:s+block] for s in starts])[:n]
        sharpes[b] = _sharpe(sample)
    lo_q, hi_q = (1-ci)/2, 1-(1-ci)/2
    lo, hi = np.quantile(sharpes, [lo_q, hi_q])
    return _sharpe(r), float(lo), float(hi)


def sharpe_ci_table(cfg: Config, dev_store: Store, hold_store: Store,
                     top_n=None, n_boot=2000, block=20, ci=0.90, verbose=True):
    """Point Sharpe and bootstrap CI for the 3 specs, development and holdout, both views."""
    top_n = top_n or cfg.top_n
    rows = []

    # --- fixed-capital view ---
    for label, store in [('development', dev_store), ('holdout', hold_store)]:
        rets, _ = _portfolio_returns_fixed(cfg, store, top_n=top_n)
        for spec, r in rets.items():
            sr, lo, hi = block_bootstrap_sharpe_ci(r, n_boot=n_boot, block=block, ci=ci)
            rows.append(dict(sample=label, view='fixed', spec=spec,
                              sharpe=sr, ci_lo=lo, ci_hi=hi, n_obs=int(np.isfinite(r).sum())))

    # --- invested view ---
    for label, store in [('development', dev_store), ('holdout', hold_store)]:
        try:
            _, curves = backtest_invested(cfg, store, top_n=top_n, verbose=False, plot=False)
        except Exception as exc:
            if verbose: print(f"[sharpe_ci] invested view not available for {label}: {exc}")
            continue
        rets = _returns_from_invested_curves(curves)
        for spec, r in rets.items():
            sr, lo, hi = block_bootstrap_sharpe_ci(r, n_boot=n_boot, block=block, ci=ci)
            rows.append(dict(sample=label, view='invested', spec=spec,
                              sharpe=sr, ci_lo=lo, ci_hi=hi, n_obs=int(np.isfinite(r).sum())))

    df = pd.DataFrame(rows)
    if verbose:
        print(f"\n{'='*78}\n SHARPE — point estimate + {int(ci*100)}% bootstrap CI (block={block}d, n_boot={n_boot})\n{'='*78}")
        print(df.round(3).to_string(index=False))
    return df

### 11.2 · Transaction-cost sensitivity
The cost enters only the P&L, so performance can be recomputed at other cost levels on the cached panel. Thresholds stay at the values calibrated with 5 bps: they are not re-tuned to the new costs, which would be in-sample again.

In [ ]:
def cost_sensitivity(cfg: Config, store: Store, top_n=None,
                      cost_grid=(0.0, 5.0, 10.0, 20.0, 40.0), verbose=True):
    """Fixed-capital portfolio metrics for each spec at several cost levels,
    with the thresholds already stored (isolates the effect of costs)."""
    top_n = top_n or cfg.top_n
    thr = store.read('params/thresholds').set_index('spec')
    qual = store.read('selection/quality')
    P, all_dates, dpos = _build_pair_panel(cfg, store)
    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    N = len(top_ids)

    rows = []
    for cost_bps in cost_grid:
        c = cost_bps/1e4
        for spec in ['ols', 'kf', 'full']:
            zs = float(thr.loc[spec, 'z_star']); ze = float(thr.loc[spec, 'z_exit'])
            S = np.zeros(len(all_dates)); POS = np.zeros(len(all_dates))
            for pid in top_ids:
                net, pos = _pair_net_returns(P[pid], spec, zs, ze, cfg, c)
                ii = np.array([dpos[d] for d in P[pid]['date']])
                S[ii] += net; POS[ii] += (pos != 0)
            r = S/max(N, 1); eq = np.cumsum(r); dd = eq - np.maximum.accumulate(eq)
            rows.append(dict(cost_bps=cost_bps, spec=spec, sharpe=_sharpe(r),
                              ann_ret=r.mean()*252, maxDD=float(dd.min()),
                              exposure=float((POS/max(N, 1)).mean())))
    df = pd.DataFrame(rows)
    if verbose:
        print(f"\n{'='*70}\n TRANSACTION-COST SENSITIVITY (thresholds frozen at cost_bps={cfg.cost_bps})\n{'='*70}")
        print(df.pivot(index='cost_bps', columns='spec', values='sharpe').round(3)
              .rename_axis(None, axis=1).to_string())
    return df


def plot_cost_sensitivity(df, title_suffix=''):
    """Net Sharpe against cost per side for the 3 specs."""
    _C = globals().get('C', {})
    col = {'ols': _C.get('normal', '#444'), 'kf': _C.get('bull', '#1f77b4'), 'full': _C.get('gfc', '#d62728')}
    lab = {'ols': 'OLS', 'kf': 'KF+GARCH', 'full': 'KF+GARCH+HMM'}
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for spec in ['ols', 'kf', 'full']:
        sub = df[df.spec == spec].sort_values('cost_bps')
        ax.plot(sub.cost_bps, sub.sharpe, 'o-', color=col[spec], label=lab[spec], lw=1.6)
    ax.axhline(0, color='k', lw=.6)
    ax.set_xlabel('Transaction cost (bps per side)')
    ax.set_ylabel('Annualized net Sharpe')
    ax.set_title(f'Transaction-cost sensitivity{title_suffix}', fontweight='bold')
    ax.legend(frameon=False)
    plt.tight_layout()
    if 'OUTPUT_DIR' in globals():
        plt.savefig(globals()['OUTPUT_DIR'] + f'cost_sensitivity{title_suffix.replace(" ", "_")}.png')
    plt.show()

### 11.3 · Sensitivity to $\phi^*$ and to the screening parameters
* `phi_star_sensitivity` — cheap: $\phi^*$ only affects `gen_positions` for `full`, so the P&L is recomputed on the cached panel.
* `screening_param_sensitivity` — expensive: `half_life_target` and `hurst_max` change the screening itself, so each grid point reruns the full development pipeline (thresholds re-optimized) in its own namespace. It is not executed below; run it once, with a small grid, to produce a robustness table. Report the whole table, not its maximum.

In [ ]:
def phi_star_sensitivity(cfg: Config, store: Store, top_n=None,
                          phi_grid=(0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80), verbose=True):
    """Fixed-capital metrics of the `full` spec across phi* values (z thresholds frozen)."""
    top_n = top_n or cfg.top_n
    thr = store.read('params/thresholds').set_index('spec')
    qual = store.read('selection/quality')
    P, all_dates, dpos = _build_pair_panel(cfg, store); c = cfg.cost_bps/1e4
    ranked = qual.nsmallest(top_n, 'rank')['pair_id'].tolist()
    top_ids = [pid for pid in ranked if pid in P]
    N = len(top_ids)
    zs = float(thr.loc['full', 'z_star']); ze = float(thr.loc['full', 'z_exit'])

    rows = []
    for phi_star in phi_grid:
        cfg_i = replace(cfg, phi_star=phi_star)   # shallow copy of Config with a new phi_star
        S = np.zeros(len(all_dates)); POS = np.zeros(len(all_dates))
        for pid in top_ids:
            net, pos = _pair_net_returns(P[pid], 'full', zs, ze, cfg_i, c)
            ii = np.array([dpos[d] for d in P[pid]['date']])
            S[ii] += net; POS[ii] += (pos != 0)
        r = S/max(N, 1); eq = np.cumsum(r); dd = eq - np.maximum.accumulate(eq)
        rows.append(dict(phi_star=phi_star, sharpe=_sharpe(r), ann_ret=r.mean()*252,
                          maxDD=float(dd.min()), exposure=float((POS/max(N, 1)).mean())))
    df = pd.DataFrame(rows)
    if verbose:
        print(f"\n{'='*60}\n phi_star SENSITIVITY (spec 'full', z thresholds frozen)\n{'='*60}")
        print(df.round(3).to_string(index=False))
    return df


def screening_param_sensitivity(base_cfg: Config, top_n=None,
                                 half_life_targets=(15.0, 25.0, 35.0),
                                 hurst_maxs=(0.40, 0.50, 0.60),
                                 verbose=True):
    """2-D grid over (half_life_target, hurst_max). Each point reruns the full
    development pipeline in its own namespace, with thresholds re-optimized, and
    records the fixed-capital Sharpe of the 3 specs. This is a robustness check on
    a design choice, not an extra parameter to maximize.
    """
    top_n = top_n or base_cfg.top_n
    rows = []
    for hl_t in half_life_targets:
        for h_max in hurst_maxs:
            ns = f"sens_hl{int(hl_t)}_hu{int(h_max*100)}"
            cfg_i = replace(base_cfg, half_life_target=hl_t, hurst_max=h_max)
            store_i = Store(cfg_i, ns=ns)
            if verbose:
                print(f"\n--- half_life_target={hl_t}  hurst_max={h_max}  (namespace={ns}) ---")
            try:
                pairs_df = screen(cfg_i, store_i, force=False, verbose=False)
                model_spreads(cfg_i, store_i, pairs_df, force=False, verbose=False)
                detect_regimes(cfg_i, store_i, force=False, verbose=False)
                make_signals(cfg_i, store_i, force=False, verbose=False)
                optimize_global(cfg_i, store_i, force=False, verbose=False)
                quality_score(cfg_i, store_i, force=False, verbose=False)
                _, aggdf = backtest(cfg_i, store_i, top_n=top_n, per_pair=False, plot=False, verbose=False)
            except Exception as exc:
                if verbose: print(f"  failed: {exc}")
                continue
            n_pairs = len(pairs_df)
            for spec in ['ols', 'kf', 'full']:
                rows.append(dict(half_life_target=hl_t, hurst_max=h_max, spec=spec,
                                  n_pairs_screened=n_pairs,
                                  sharpe=aggdf.loc[spec, 'sharpe'],
                                  maxDD=aggdf.loc[spec, 'maxDD']))
    df = pd.DataFrame(rows)
    if verbose:
        print(f"\n{'='*70}\n SENSITIVITY half_life_target x hurst_max (fixed-capital view, all 3 specs)\n{'='*70}")
        print(df.round(3).to_string(index=False))
    return df

### 11.4 · Screening audit
`screen` silently skips pairs that fail a test or raise an exception. `screen_diagnostic` replays the same funnel read-only and counts the rejection reasons, separating economic rejections from numerical failures (the `*_exception` rows).

> **How to read the counts.** `not_I1` is counted per **ticker**, all other reasons per **pair**; the rows therefore do not sum to the number of candidate pairs. `ok` counts pairs that pass every filter *before* the per-bucket, per-window and per-asset caps of `cap_window`, so it is larger than the number of pairs finally admitted by `screen`. The audit covers the windows of the store it is given (development only in the call below).

In [ ]:
from collections import Counter

def tradability_diag(y1, y2, cfg):
    """Same filters as tradability(), but also returns the rejection reason."""
    y1, y2 = np.asarray(y1, float), np.asarray(y2, float)
    rho = np.corrcoef(np.diff(y1), np.diff(y2))[0, 1]
    if not np.isfinite(rho):
        return None, 'corr_nan'
    if rho > cfg.corr_max:
        return None, 'corr_too_high'
    beta = np.polyfit(y2, y1, 1)[0]; spread = y1 - beta*y2
    hl = half_life(spread, len(spread))
    if not np.isfinite(hl):
        return None, 'half_life_nonstationary'
    if not (cfg.half_life_min <= hl <= cfg.half_life_max):
        return None, 'half_life_out_of_range'
    H = hurst(spread)
    if H > cfg.hurst_max:
        return None, 'hurst_too_high'
    VR = variance_ratio(spread)
    if VR > cfg.vr_max:
        return None, 'variance_ratio_too_high'
    score = abs(np.log(hl/cfg.half_life_target)) + H + VR
    return dict(beta=beta, rho=rho, half_life=hl, hurst=H, vr=VR, score=score), 'ok'


def screen_diagnostic(cfg: Config, store: Store, holdout=False, max_windows=None, verbose=True):
    """Read-only replay of the screen() funnel that counts rejection reasons.
    Nothing is written to the store. `max_windows` limits the number of windows
    (useful for a quick check before a full run)."""
    close, groups0, vol = store.load_universe()
    logclose = np.log(close)
    reasons = Counter()
    n_candidates_total = 0
    windows_seen = 0

    for (f0, f1, e1, t1) in walk_forward(close.index, cfg, holdout=holdout):
        if max_windows is not None and windows_seen >= max_windows:
            break
        windows_seen += 1
        close_w, log_w = close.loc[f0:f1], logclose.loc[f0:f1]
        vol_w = vol.loc[f0:f1] if vol is not None else None
        univ = quality_filter(close_w, vol_w, cfg)
        groups = build_groups(close_w, univ, groups0, cfg)

        for t in (x for g in groups.values() for x in g):
            try:
                ok = is_I1(log_w[t], cfg)
            except Exception:
                reasons['is_I1_exception'] += 1
                continue
            if not ok:
                reasons['not_I1'] += 1

        i1 = {t: is_I1(log_w[t], cfg) for g in groups.values() for t in g}
        for g, ts in groups.items():
            if g in NO_PAIR_BUCKETS: continue
            ts1 = [t for t in ts if i1.get(t, False)]
            if len(ts1) < 2: continue
            cand = list(combinations(ts1, 2))
            n_candidates_total += len(cand)
            aligned = {}
            for a, b in cand:
                try:
                    aligned[(a, b)] = align_pair(log_w, a, b)
                except Exception:
                    reasons['align_exception'] += 1
            valid_cand = [ab for ab in cand if ab in aligned]
            pvals = []
            for ab in valid_cand:
                try:
                    pvals.append(eg_pvalue(*aligned[ab]))
                except Exception:
                    reasons['eg_pvalue_exception'] += 1
                    pvals.append(1.0)
            mask = benjamini_hochberg(pvals, cfg.fdr_q) if pvals else np.array([], bool)
            for (a, b), pv, ok in zip(valid_cand, pvals, mask):
                if not ok:
                    reasons['fdr_rejected'] += 1
                    continue
                ya, yb = aligned[(a, b)]
                try:
                    stable = is_stable(ya, yb, cfg.stability_alpha)
                except Exception:
                    reasons['is_stable_exception'] += 1
                    continue
                if not stable:
                    reasons['not_stable_split_sample'] += 1
                    continue
                try:
                    tr, reason = tradability_diag(ya, yb, cfg)
                except Exception:
                    reasons['tradability_exception'] += 1
                    continue
                reasons[reason] += 1

    df = pd.DataFrame(sorted(reasons.items(), key=lambda kv: -kv[1]), columns=['reason', 'count'])
    if verbose:
        print(f"\n{'='*60}\n SCREENING AUDIT — rejection reasons ({windows_seen} windows, "
              f"{n_candidates_total} candidate pairs)\n{'='*60}")
        print(df.to_string(index=False))
        exc_reasons = [r for r in reasons if r.endswith('_exception')]
        n_exc = sum(reasons[r] for r in exc_reasons)
        if n_exc > 0:
            print(f"\n⚠ {n_exc} rejections caused by EXCEPTIONS rather than economic criteria — "
                  f"worth investigating: {[r for r in exc_reasons]}")
    return df

### 11.5 · Survivorship-bias audit
Without a point-in-time constituent database the bias cannot be corrected, but it can be detected:

* **(A)** in a genuine historical panel many tickers stop trading well before the end of the sample (bankruptcies, M&A, delistings);
* **(B)** well-known names that left the index should appear in the early windows.

If almost no ticker "dies" and none of the known names is present, the universe has been built from current survivors filled backwards.

In [ ]:
# Sample of large S&P 500 names that disappeared through bankruptcy, acquisition or
# delisting in known years (diagnostic only, not exhaustive).
_KNOWN_DELISTED_OR_ACQUIRED = {
    'ENE':  ('Enron', 2001),               # bankruptcy 2001
    'WCOEQ':('WorldCom', 2002),            # bankruptcy 2002 (post-bankruptcy ticker)
    'LEH':  ('Lehman Brothers', 2008),     # bankruptcy 2008
    'BSC':  ('Bear Stearns', 2008),        # acquired by JPM 2008
    'CPQ':  ('Compaq', 2002),              # merged into HP 2002
    'MCIC': ('MCI Communications', 1998),  # acquired by WorldCom 1998
    'GTE':  ('GTE Corp', 2000),            # merged into Verizon 2000
    'BUD':  ('Anheuser-Busch (old)', 2008),# acquired by InBev 2008
    'WB':   ('Wachovia', 2008),            # acquired by Wells Fargo 2008
    'CFC':  ('Countrywide Financial', 2008), # acquired by BofA 2008
    'S':    ('Sears Holdings (old)', 2018),  # bankruptcy 2018 (ticker later reused by SentinelOne)
}


def survivorship_bias_audit(cfg: Config, verbose=True):
    """Checks (A) and (B) on the main price database."""
    store = Store(cfg)
    close, groups0, vol = store.load_universe()

    # --- (A) tickers that stop trading mid-sample ---
    panel_end = close.index.max()
    last_valid = close.apply(lambda s: s.last_valid_index())
    # a ticker 'dies mid-sample' if its last valid observation is more than
    # 2 years before the end of the panel
    cutoff = panel_end - pd.DateOffset(years=2)
    died_mid_sample = last_valid[last_valid < cutoff]
    frac_died = len(died_mid_sample) / max(len(last_valid), 1)

    # --- (B) presence of known delisted names ---
    present = {t: info for t, info in _KNOWN_DELISTED_OR_ACQUIRED.items() if t in close.columns}
    absent = {t: info for t, info in _KNOWN_DELISTED_OR_ACQUIRED.items() if t not in close.columns}

    if verbose:
        print(f"\n{'='*72}\n SURVIVORSHIP-BIAS AUDIT\n{'='*72}")
        print(f"Panel: {close.shape[1]} tickers, {close.index.min().date()} -> {panel_end.date()}\n")
        print(f"(A) Tickers whose last valid observation is before {cutoff.date()} "
              f"(died mid-sample): {len(died_mid_sample)}/{len(last_valid)} "
              f"({frac_died:.1%})")
        if frac_died < 0.02:
            print("    ⚠ STRONG SUSPICION: almost no ticker 'dies' before the end of the "
                  "sample. A genuine historical universe (bankruptcies, M&A, delistings over "
                  "30 years) should show a clearly positive share. Check how the database "
                  "was built -- survivorship bias is likely.")
        else:
            print(f"    ✓ A non-negligible share of tickers ends before the sample end -- "
                  f"consistent with a genuine historical panel, but not conclusive "
                  f"(check point B as well).")

        print(f"\n(B) Known names that disappeared through bankruptcy/M&A, present in the universe: "
              f"{len(present)}/{len(_KNOWN_DELISTED_OR_ACQUIRED)}")
        for t, (name, year) in present.items():
            print(f"    ✓ {t} ({name}, gone ~{year}) present")
        for t, (name, year) in absent.items():
            print(f"    ✗ {t} ({name}, gone ~{year}) MISSING")
        if len(present) == 0:
            print("\n    ⚠ STRONG SUSPICION: none of the known delisted names is present. "
                  "The universe was very likely backfilled from current survivors only.")

    return dict(frac_died_mid_sample=frac_died, n_died=len(died_mid_sample),
                known_present=list(present), known_absent=list(absent))

## 12 · Experiments
Set `hdf_file` to your copy of the database. `top_n=50` is the Top-50 portfolio used in the thesis. The first call runs every stage from scratch (slow); later calls reuse the cache.

### 12.1 · Development sample (trading windows ending by 2019-12-31)

In [ ]:
cfg = Config(hdf_file='/content/drive/MyDrive/data_ip_2026_v2.h5', top_n=5)
per_pair_dev, agg_dev = run_pipeline(cfg, top_n=50)
print('\n=== DEVELOPMENT TOP-50 ==='); print(agg_dev.round(3).to_string())

### 12.2 · Holdout (trading windows ending from 2020-01-01), thresholds frozen
First list the holdout windows, then run the holdout chain once.

In [ ]:
_ = preview_holdout_windows(cfg)   # no-look-ahead check

In [ ]:
per_pair_h, agg_h = run_holdout(cfg, force=True, top_n=50)
print('\n=== HOLDOUT TOP-50 ==='); print(agg_h.round(3).to_string())

Holdout broken down by trading window (flat view):

In [ ]:
hold = Store(cfg, ns='holdout')
by_win_h = backtest_by_window(cfg, hold, top_n=50)

### 12.3 · Invested view (per-window allocation, chained equity)

In [ ]:
res_inv_dev, curves_dev = backtest_invested(cfg, Store(cfg), top_n=50, plot=True)
print(res_inv_dev.round(4).to_string())

In [ ]:
res_inv_h, curves_h = backtest_invested(cfg, Store(cfg, ns='holdout'), top_n=50, plot=True)
print(res_inv_h.round(4).to_string())

### 12.4 · Development vs holdout figure
Main figure: invested view. Secondary figure: fixed-capital view, where exposure is diluted across pairs that are not trading, so its Sharpe ratios and drawdowns must be read with care.

In [ ]:
from matplotlib.patches import Patch

specs  = ['ols', 'kf', 'full']; labels = ['OLS', 'KF+GARCH', 'KF+GARCH+HMM']
bar_c  = [C['normal'], C['bull'], C['gfc']]

def paired(ax, dev, hold, ylabel, title):
    """Side-by-side bars: development (plain) vs holdout (hatched)."""
    x = np.arange(3); w = 0.38
    ax.bar(x - w/2, dev,  w, color=bar_c, alpha=0.85, edgecolor='k', linewidth=0.6)
    ax.bar(x + w/2, hold, w, color=bar_c, alpha=0.85, edgecolor='k', linewidth=0.6, hatch='///')
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.axhline(0, color='k', lw=.7)
    ax.set_ylabel(ylabel); ax.set_title(title)

# --- main figure: invested view ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
paired(axes[0], res_inv_dev.loc[specs, 'sharpe'].values, res_inv_h.loc[specs, 'sharpe'].values,
       'Sharpe (annualized)', 'Net Sharpe: development vs holdout (invested view)')
paired(axes[1], 100*res_inv_dev.loc[specs, 'maxDD'].values, 100*res_inv_h.loc[specs, 'maxDD'].values,
       'max drawdown (%)', 'Max drawdown: development vs holdout (invested view)')
axes[0].legend(handles=[Patch(facecolor='0.7', edgecolor='k', label='Development (≤2019)'),
                        Patch(facecolor='0.7', edgecolor='k', hatch='///', label='Holdout (2020+)')],
               frameon=False, loc='upper right')
fig.suptitle('KF–GARCH–HMM vs OLS: development vs holdout (invested view)',
             fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(OUTPUT_DIR + 'dev_vs_holdout_invested.png'); plt.show()


# --- secondary figure: fixed-capital 1/N view (diluted exposure, read with care) ---
fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4))
paired(axes2[0], agg_dev.loc[specs, 'sharpe'].values, agg_h.loc[specs, 'sharpe'].values,
       'Sharpe (annualized)', 'Net Sharpe — fixed-capital 1/N view')
paired(axes2[1], 100*agg_dev.loc[specs, 'maxDD'].values, 100*agg_h.loc[specs, 'maxDD'].values,
       'max drawdown (%)', 'Max drawdown — fixed-capital 1/N view')
axes2[0].legend(handles=[Patch(facecolor='0.7', edgecolor='k', label='Development (≤2019)'),
                         Patch(facecolor='0.7', edgecolor='k', hatch='///', label='Holdout (2020+)')],
                frameon=False, loc='upper right')
fig2.suptitle('Robustness check: fixed-capital 1/N view (diluted exposure)',
              fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(OUTPUT_DIR + 'dev_vs_holdout_fixed_secondary.png'); plt.show()

### 12.5 · Bootstrap confidence intervals for the Sharpe ratios

In [ ]:
sharpe_ci = sharpe_ci_table(cfg, Store(cfg), Store(cfg, ns='holdout'), top_n=50)
sharpe_ci.to_csv(OUTPUT_DIR + 'sharpe_bootstrap_ci.csv', index=False)

### 12.6 · Transaction-cost sensitivity

In [ ]:
dev_costs = cost_sensitivity(cfg, Store(cfg), top_n=50)
plot_cost_sensitivity(dev_costs, ' — Development')

In [ ]:
hold_costs = cost_sensitivity(cfg, Store(cfg, ns='holdout'), top_n=50)
plot_cost_sensitivity(hold_costs, ' — Holdout')

### 12.7 · Sensitivity to $\phi^*$ (development)

In [ ]:
phi_df = phi_star_sensitivity(cfg, Store(cfg), top_n=50)

### 12.8 · Screening audit (development windows)

In [ ]:
audit_screening = screen_diagnostic(cfg, Store(cfg))

### 12.9 · Second holdout: the 2008–2009 crisis
Independent development sample ending on 2007-06-30, thresholds frozen, single test on 2008–2009. The basket notebook reuses the thresholds written here (namespace `crisis2008_dev`).

In [ ]:
agg_pre_crisis_dev, agg_crisis_hold = run_crisis_holdout_2008(
    cfg,
    pre_crisis_dev_end='2007-06-30',
    crisis_start='2008-01-01',
    crisis_end='2009-12-31',
    top_n=50,
)

In [ ]:
two_crises_table = compare_two_crises(agg_dev, agg_h, agg_pre_crisis_dev, agg_crisis_hold)
two_crises_table.to_csv(OUTPUT_DIR + 'two_crises_comparison.csv')

### 12.10 · Survivorship-bias audit

In [ ]:
audit_bias = survivorship_bias_audit(cfg)